# UR5e 焊接机械臂摆动施焊 — NVIDIA RTX 实时交互演示

本 notebook 继承 `robot6_weave_interactive_demo.ipynb` 的完整计算链，
但把所有交互式三维场景换成 `vendor/trame-rtx-widget` 提供的
`RTXLiveWidget` 前端和原生 VTK `FrameScene`：

- **机械臂**：`SixDofArm`（模块 7b）按 **Universal Robots UR5e**
  参数化（`conf/model/robot6_ur5e.yaml`）。连杆、关节和焊枪仍直接
  来自 `arm._kin(q)` 的同一份 DH 运动学；§2/§3 用服务端 RTX 图像流
  显示静态姿态和 41 帧可播放轨迹。选型按 UR3e（500 mm）/
  **UR5e（850 mm）**/UR10e（1300 mm）臂展比较：焊缝位于基座前
  450 mm，UR5e 与原默认 850 mm 球腕臂工作区相当。标准 DH 表和
  连杆质量来自 UR 参数表；`r_link` / `J_rotor` 仍是本模型的圆柱惯量
  与折算转子惯量近似，不冒充官方数据。
- **摆动施焊**：数据库中位工况 + 2 Hz × 4 mm 三角摆；
  `track_path` 强迫 DEL 跟踪，实际 TCP 轨迹经
  `RobotExecutedWeave` 注入 `GoldakFDM`。§4 使用 `xfine`
  传导网格，§4b 叠加 10A 有效导热率对流修正。
- **原生热场管线**：Goldak 网格写入持久 `vtkStructuredGrid`，
  标量按 `ravel(order="F")` 映射；熔池、HAZ、切片由
  `vtkContourFilter` / `vtkCutter` 计算。§5b 动画只更新原有
  NumPy/VTK 标量数组，允许等值面拓扑逐帧变化。
- **前端能力**：工具栏提供播放、帧滑块和相机复位；图层菜单控制
  机械臂、工件、轨迹、瞬时/峰值热场及对流叠加。普通拖动控制相机；
  点击机械臂或工艺对象可按逻辑组选择（黄框），`Shift + 左拖` 平移、
  `Ctrl + 左拖` 绕组中心旋转。§5 的两个 live 页还可左拖橙色 TCP
  手柄，以 Mink IK 固定工具姿态并实时改变 UR5e 构型；手柄可在图层
  菜单隐藏，播放、换帧或重置对象位姿会恢复记录构型。原 notebook
  的可复现视图/机械臂透明度
  滑块仍保留，并直接
  改动同一 RTX 场景，不再重建 iframe。坐标轴标签使用世界坐标中的
  camera-facing 文字，始终随三维场景缩放。每个视图还共用程序化天空、
  有限网格地坪及 ambient + diffuse 摄影棚灯组。
- **动画输出**：§5b 仍写出
  `results/robot6_weave_seam.gif`，但帧来自同一个已验证的
  NVIDIA EGL 渲染窗；随后同一组快照也可在 RTX 前端实时播放。

## 前置条件

这是**服务端 NVIDIA RTX/EGL 光栅渲染**，不是浏览器 WebGL 或
软件渲染回退。`NvidiaEGLContext.verify()` 会检查 EGL/OpenGL
vendor 和 renderer，并在不是 NVIDIA RTX 时直接失败。

```bash
git submodule update --init vendor/trame-rtx-widget
uv sync --extra rtx --locked
uv run --extra rtx --locked python -c "import vtk; print(hasattr(vtk, 'vtkEGLRenderWindow'))"
uv run --extra rtx --locked jupyter lab
```

`rtx` extra 已通过 `tool.uv.sources` 把 vendored widget 作为 editable
path dependency 安装，并统一解析其 Trame/VTK 版本；同步后请重启内核。
它与普通 PyVista `notebook` extra 使用互斥的 Trame 后端版本，因此不要
同时选择两者；本 notebook 只需 `--extra rtx`。
能力检查必须输出 `True`，机器还需有可用的 NVIDIA RTX 驱动。
若当前内核没有 EGL/RTX（例如 macOS），第一个代码格会明确标记原因，
随后在同一内核中运行 `robot6_weave_interactive_demo.ipynb` 的 PyVista
前端；余下 RTX 专用代码格会自动跳过，不再以异常中断 Run All。
远程 Jupyter 若通过反向代理访问，可在启动内核前配置
`TRAME_IFRAME_BUILDER=serverproxy`。VS Code 的受限 webview 通常
无法连接 Trame 的实时服务器，推荐浏览器中的 JupyterLab。

全程约 3–4 分钟（跟踪仿真 ~1 min + xfine 传导场 ~30 s +
fine 对流场 ~20–30 s + §5b 求解/RTX GIF ~1 min）。


## 0. 连接本地 RTX widget 并验证 EGL 后端

项目的 `rtx` extra 从仓库的 `vendor/trame-rtx-widget` editable
path dependency 导入 widget。这里不修改 `sys.path`，也不调用
`ensure_display()`、Xvfb 或
`pv.set_jupyter_backend()`：RTX 场景必须由显式
`vtkEGLRenderWindow` 拥有，且不会把 X11、OSMesa、Cocoa 或 vtk.js
冒充 RTX。缺少 EGL/RTX 时会显式切换到独立的 PyVista notebook，
其页面和 backend label 均保留 PyVista 名称。

每个 live widget 使用独立 Trame server 和 EGL context；重跑场景格
时会先 `await old_widget.close()`，释放旧 server、remote view 和
GPU 资源。


In [ ]:
import json
import os
import platform
import sys
from html import escape
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import vtk
from IPython.display import HTML, display
from packaging.version import Version


def find_repo_root():
    """从 Jupyter 当前目录向上寻找 welding-dynamics 仓库。"""
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "pyproject.toml").is_file() and (
            base / "notebooks" / "robot6_weave_interactive_demo.ipynb"
        ).is_file():
            return base
    raise FileNotFoundError(
        "找不到 welding-dynamics 仓库根目录；请从仓库内启动 Jupyter。"
    )


REPO_ROOT = find_repo_root()
RTX_WIDGET_SRC = REPO_ROOT / "vendor" / "trame-rtx-widget" / "src"
PORTABLE_NOTEBOOK = (
    REPO_ROOT / "notebooks" / "robot6_weave_interactive_demo.ipynb"
)
_portable_document = json.loads(PORTABLE_NOTEBOOK.read_text())
_PORTABLE_CODE_CELLS = tuple(
    "".join(cell["source"])
    for cell in _portable_document["cells"]
    if cell["cell_type"] == "code"
)
if len(_PORTABLE_CODE_CELLS) != 8:
    raise RuntimeError(
        "PyVista fallback 必须保持 8 个与 RTX 对齐的代码格；"
        f"当前为 {len(_PORTABLE_CODE_CELLS)} 个。"
    )

RTX_BACKEND_AVAILABLE = False
RTX_FALLBACK_REASON = None
RTX_FALLBACK_ERROR = None
_probe = None
if not hasattr(vtk, "vtkEGLRenderWindow"):
    RTX_FALLBACK_REASON = (
        f"{platform.system()} {platform.machine()} 的 VTK "
        f"{vtk.vtkVersion.GetVTKVersion()} 没有 vtkEGLRenderWindow"
    )
else:
    _minimums = {
        "trame-rtx-widget": "0.1.0",
        "trame": "3.13.2",
        "trame-vtk": "2.11.15",
        "trame-vuetify": "3.2.5",
        "vtk": "9.4.0",
        "mink": "1.2.0",
    }
    _dependency_issues = []
    for name, minimum in _minimums.items():
        try:
            installed = version(name)
        except PackageNotFoundError:
            _dependency_issues.append(f"{name} 未安装")
        else:
            if Version(installed) < Version(minimum):
                _dependency_issues.append(
                    f"{name} {installed} < {minimum}"
                )
    if _dependency_issues:
        raise RuntimeError(
            "RTX widget 环境未同步："
            + ", ".join(_dependency_issues)
            + "。请在仓库根目录运行 "
              "`uv sync --extra rtx --locked`，然后重启 Jupyter 内核。"
        )

    from trame_rtx_widget import (  # noqa: F401
        LayerSpec,
        NvidiaEGLContext,
        RTXLiveWidget,
    )
    from vtk.util.numpy_support import (  # noqa: F401
        numpy_to_vtk,
        vtk_to_numpy,
    )

    try:
        _probe = NvidiaEGLContext(size=(32, 32), multisamples=0)
        _probe_info = _probe.verify()
    except Exception as exc:  # noqa: BLE001 -- capability boundary
        RTX_FALLBACK_REASON = f"{type(exc).__name__}: {exc}"
    else:
        RTX_BACKEND_AVAILABLE = True
    finally:
        if _probe is not None:
            _probe.close()


def _run_portable_cell(index):
    """Execute the matching PyVista cell with notebook-relative paths."""
    source = _PORTABLE_CODE_CELLS[int(index)]
    original_cwd = Path.cwd()
    notebook_dir = str(PORTABLE_NOTEBOOK.parent)
    sys.path.insert(0, notebook_dir)
    try:
        os.chdir(notebook_dir)
        result = get_ipython().run_cell(source)
    finally:
        os.chdir(original_cwd)
        sys.path.remove(notebook_dir)
    result.raise_error()
    return result.result


def _run_rtx_only(line, cell):
    """Dispatch one section to RTX or its matching PyVista cell."""
    if RTX_BACKEND_AVAILABLE:
        result = get_ipython().run_cell(cell)
        result.raise_error()
        return result.result
    if RTX_FALLBACK_ERROR is not None:
        return None
    return _run_portable_cell(int(line.strip()))


get_ipython().register_magic_function(
    _run_rtx_only, magic_kind="cell", magic_name="rtx_only"
)

if RTX_BACKEND_AVAILABLE:
    print(
        f"vendor source: {RTX_WIDGET_SRC}\n"
        f"VTK {vtk.vtkVersion.GetVTKVersion()} · {_probe_info.label}"
    )
else:
    _reason_html = escape(str(RTX_FALLBACK_REASON))
    display(HTML(
        "<div style='padding:12px 14px;border:1px solid #d4a72c;"
        "border-radius:8px;background:#fff8c5'>"
        "<b>NVIDIA RTX/EGL unavailable — using PyVista fallback.</b>"
        f"<br><code>{_reason_html}</code><br>"
        "The following output is rendered by the portable PyVista "
        "frontend, not RTX. "
        "<a href='robot6_weave_interactive_demo.ipynb' target='_blank'>"
        "Open the PyVista notebook</a>.</div>"
    ))
    try:
        _run_portable_cell(0)
    except Exception as exc:  # noqa: BLE001 -- notebook handoff boundary
        RTX_FALLBACK_ERROR = exc
        display(HTML(
            "<div style='padding:12px 14px;border:1px solid #cf222e;"
            "border-radius:8px;background:#ffebe9'>"
            "<b>PyVista fallback could not start.</b><br><code>"
            f"{escape(type(exc).__name__ + ': ' + str(exc))}</code><br>"
            "Run <code>uv sync --extra notebook --locked</code>, "
            "restart the kernel, then open the linked notebook."
            "</div>"
        ))


## 1. 跟踪仿真: 数据库中位工况 + 2 Hz × 4 mm 三角摆

与《机器人部署场景》同一设置: 焊缝沿机器人基座 x 轴 (工作空间中心区,
平焊, 枪尖竖直向下), 参考轨迹 = 恒速焊缝 + 摆动指令, `track_path`
强迫 DEL 全动力学跟踪 5 s。UR 偏置腕没有球腕解耦, 位姿逆解仍走同一
DLS 数值 IK — 唯一要额外指定的是**解支**: `Q_SEED` 选肘上抬、基座朝前
的典型 UR 作业姿态 (随机种子会落到卷绕/肘下解支)。另按相同参考做一遍
**运动学 IK 采样** (251 帧), 供后面渲染机械臂姿态 — 实际关节状态与之
只差亚毫米, 在场景尺度上不可分辨。

In [2]:
%%rtx_only 1
import numpy as np
from hydra import compose, initialize_config_module
from hydra.utils import instantiate

from welding_dynamics.config import arc_power   # 导入即注册 wd.* 解析器
from welding_dynamics import RobotExecutedWeave


def compose_cfg(config_name, *overrides):
    with initialize_config_module(config_module="welding_dynamics.conf",
                                  version_base="1.3"):
        return compose(config_name=config_name, overrides=list(overrides))


# UR5e 参数化 (model@robot6=robot6_ur5e); CLI 等价 welding-sim-vi model@robot6=robot6_ur5e
# solver=xfine (0.5 mm): 摆动节距 v/f ≈ 2.6 mm 需 ~5 格才不出夸大的"鱼鳞"棱片
# (fine 的 0.8 mm 下仅 ~3 格, 接近网格 Nyquist, 棱脊间距被网格拍频调制成 1.6–4 mm)
arm = instantiate(compose_cfg("sim_vi", "model@robot6=robot6_ur5e").robot6)
cfg = compose_cfg("sim_3d", "process=db_median", "solver=xfine")
weave = instantiate(compose_cfg("sim_3d", "weave=triangle").weave)
V_WELD = float(cfg.process.travel_speed_m_s)          # 5.15 mm/s
Q_ARC = arc_power(cfg)                                # 8120 W (U·I)
P0 = np.array([0.45, 0.0, 0.25])                      # m 焊缝起点 (板顶面)
T_TRACK = float(cfg.solver.t_end)                     # 5 s
R_REF = np.diag([1.0, -1.0, -1.0])                    # 平焊: 枪尖竖直向下
Q_SEED = (0.3, -2.0, -1.6, 2.1, -1.57, 1.9)           # UR 肘上抬/基座朝前解支

print(f"UR5e: 连杆质量 {[float(x) for x in arm.m]} kg, reach 850 mm, "
      f"|a2|+|a3| = {(abs(arm.dh_a[1]) + abs(arm.dh_a[2]))*1e3:.0f} mm, "
      f"偏置腕 d5 = {arm.dh_d[4]*1e3:.1f} mm")


def p_ref(t):
    dx, dy = weave.offset(t)
    return P0 + np.array([V_WELD*t + dx, dy, 0.0])


t_tr, tip, ref, err = arm.track_path(p_ref, T_TRACK, q_seed=Q_SEED)
print(f"跟踪 {weave.describe()}: 三维 RMS {1e3*np.sqrt((err**2).mean()):.2f} mm, "
      f"执行横向峰-峰 {1e3*np.ptp(tip[:, 1]):.2f} mm (指令 4.00 mm)")

# 渲染用姿态帧: 沿参考轨迹的 IK 采样 (热启动保持解支连续)
DTQ = T_TRACK/250
q_grid, q_seed = [], Q_SEED
for tk in np.arange(251)*DTQ:
    q_seed = arm.ik(p_ref(tk), R_REF, q0=q_seed)
    q_grid.append(q_seed)
q_grid = np.array(q_grid)


def q_at(t):
    return q_grid[min(250, max(0, int(round(t/DTQ))))]


print(f"姿态帧: {len(q_grid)} 帧 @ {DTQ*1e3:.0f} ms")

UR5e: 连杆质量 [3.761, 8.058, 2.846, 1.37, 1.3, 0.365] kg, reach 850 mm, |a2|+|a3| = 817 mm, 偏置腕 d5 = 99.7 mm
跟踪 三角(直线摆) 2 Hz × 4 mm: 三维 RMS 0.69 mm, 执行横向峰-峰 4.09 mm (指令 4.00 mm)
姿态帧: 251 帧 @ 20 ms


## 2. 机械臂场景（焊接中途 t = 2.5 s）

连杆圆柱、关节球、焊枪锥仍由 `arm._kin(q)` 的帧原点直接生成，
渲染几何与动力学模型共享同一份 UR5e DH 表；偏置腕的 wrist-1/2/3
三段腕链在枪头附近清晰可见。红色细管是实际执行 TCP 轨迹，黑线是
焊缝中心线。

本格定义后续场景共用的原生 VTK 工具箱：

- `WeldingRTXSceneBase` 实现 vendor 的 `FrameScene` 协议、显式
  NVIDIA EGL context、图层/模式可见性、相机 home/reset 和安全释放；
- `RobotRig` 只在初始化时创建 link/joint/gun actor，换帧时更新
  `vtkLineSource`、球 actor 位置和 `vtkConeSource`；
- `PolylineRig` 原位更新不断增长的 TCP 轨迹；
- `vtkStructuredGrid`、等值面、切片和标尺直接进入同一个 RTX renderer。

RTX 工具栏的图层按钮可隐藏机械臂以完全透视焊缝；下面滑块保留
原 notebook 的平移/旋转/臂透明度功能，值继续写入
`.robot6_view_state.json`。鼠标视角在换帧/切换图层时保持，camera
reset 回到滑块定义的可复现视角。点击对象显示黄色选择框；按住
`Shift` 左拖可平移完整机械臂或全部工艺对象，按住 `Ctrl` 左拖可绕
所选组中心旋转。普通左拖/中拖/右拖（或滚轮）继续控制相机。每个
RTX 控制区还提供“重置对象位姿”按钮，播放中开始对象拖动会自动暂停。

§4 尚未运行时 caption 会提示先求热场；§4 完成后会把 `g.peak`
熔合区历史和 `g.T` 瞬时熔池原位接入本场景。图层菜单的
“instantaneous field” 开关在二者之间切换，不会移动相机；若在新内核
中先跑了 §4，也可重跑本格直接带热场创建。


In [3]:
%%rtx_only 2
import json

import ipywidgets as widgets
import numpy as np
from IPython.display import display
from matplotlib import colormaps
from matplotlib.colors import to_rgb
from welding_dynamics.robot_ik import MinkArmIK


def close_weld_output_tabs():
    """关闭 §5/§5b 的 tab 外壳，但不误关仍可复用的 live app。"""
    for name in ("composite_rtx", "seam_rtx"):
        app = globals().get(name)
        if (
            app is not None
            and not getattr(app, "_closed", True)
            and bool(app.state.playing)
        ):
            with app.state:
                app.state.playing = False
    tabs = globals().get("weld_output_tabs")
    if tabs is not None:
        handler = getattr(
            tabs, "_rtx_selection_handler",
            globals().get("_on_weld_output_tab"),
        )
        if handler is not None:
            try:
                tabs.unobserve(handler, names="selected_index")
            except ValueError:
                pass
        # Tab.close() 不会递归关闭 children；先解除 live panel 所有权。
        tabs.children = ()
        tabs.close()
    globals()["_on_weld_output_tab"] = None
    for name in ("weld_gif_image", "weld_gif_caption", "weld_gif_panel"):
        owned = globals().get(name)
        if owned is not None:
            owned.close()
        globals()[name] = None
    globals()["weld_output_tabs"] = None

# 若本格被重跑，先释放下游可能仍存活的 Trame server / EGL context。
close_weld_output_tabs()
for _live_name in ("arm_rtx", "pose_rtx", "composite_rtx", "seam_rtx"):
    _old_live = globals().get(_live_name)
    if _old_live is not None:
        _old_scene = getattr(_old_live, "scene", None)
        await _old_live.close()
        globals()[_live_name] = None
        _scene_name = _live_name.removesuffix("_rtx") + "_scene"
        if globals().get(_scene_name) is _old_scene:
            globals()[_scene_name] = None

R_LINK = [45.0, 38.0, 32.0, 22.0, 20.0, 16.0]  # mm
R_JOINT = [50.0, 42.0, 36.0, 26.0, 24.0]       # mm
TIP_HANDLE_HIT_RADIUS_PX = 24.0
VIEW_KEYS = ("px", "py", "pz", "az", "el", "roll", "arm_t")
VIEW_STATE = REPO_ROOT / "notebooks" / ".robot6_view_state.json"
try:
    _view_state = json.loads(VIEW_STATE.read_text())
except Exception:
    _view_state = {}


def state_value(scene, name, default):
    return _view_state.get(scene, {}).get(name, default)


def save_state(scene, name, value):
    _view_state.setdefault(scene, {})[name] = value
    VIEW_STATE.write_text(json.dumps(_view_state, indent=1,
                                     ensure_ascii=False))


def tracked(scene, name, control):
    value = state_value(scene, name, None)
    if value is not None:
        control.value = value
    def persist(change):
        save_state(scene, name, change["new"])
    control.observe(persist, names="value")
    control._rtx_persist_handler = persist
    return control


def view_widgets(scene, lim=300.0, step=10.0):
    """原 notebook 的 3 平移 + 3 旋转 + 机械臂透明度控件。"""
    common = dict(
        continuous_update=False,
        readout_format=".0f",
        layout=widgets.Layout(width="230px"),
    )
    pans = [
        tracked(
            scene,
            f"p{axis}",
            widgets.FloatSlider(
                min=-lim, max=lim, step=step, value=0.0,
                description=f"平移 {axis} [mm]", **common,
            ),
        )
        for axis in "xyz"
    ]
    rotations = [
        tracked(
            scene,
            key,
            widgets.FloatSlider(
                min=lo, max=hi, step=2.0, value=0.0,
                description=f"{label} [°]", **common,
            ),
        )
        for key, label, lo, hi in (
            ("az", "方位角", -180.0, 180.0),
            ("el", "俯仰角", -80.0, 80.0),
            ("roll", "滚转", -180.0, 180.0),
        )
    ]
    transparency = tracked(
        scene,
        "arm_t",
        widgets.FloatSlider(
            min=0.0, max=100.0, step=5.0, value=0.0,
            description="臂透明 [%]", **common,
        ),
    )
    return pans + rotations + [transparency]


def layer_spec(scene, key, label, visible=True):
    storage = {"instant_mode": "inst", "convection": "conv"}.get(
        key, f"layer_{key}"
    )
    return LayerSpec(
        key, label, visible=bool(state_value(scene, storage, visible))
    )


def finished_pool():
    """返回已经完成 run() 的 g；中断求解留下的半成品不接入场景。"""
    candidate = globals().get("g")
    required = ("peak", "T", "Tm", "x", "y", "z")
    return candidate if all(hasattr(candidate, key) for key in required) else None


def _rgb(color):
    return tuple(float(channel) for channel in to_rgb(color))


def vtk_rgb_image(rgb):
    """把 H×W×3 uint8 图像转换为自持有的 VTK RGB image。"""
    rgb = np.ascontiguousarray(rgb, dtype=np.uint8)
    height, width, channels = rgb.shape
    if channels != 3:
        raise ValueError("RGB image must have exactly three channels")
    image = vtk.vtkImageData()
    image.SetDimensions(width, height, 1)
    image.GetPointData().SetScalars(numpy_to_vtk(
        rgb.reshape(-1, 3),
        deep=True,
        array_type=vtk.VTK_UNSIGNED_CHAR,
    ))
    return image


def make_sky_texture(width=256, height=128):
    """生成无外部资源的 equirectangular 天空与柔和日光。"""
    longitude = np.linspace(-np.pi, np.pi, int(width))[None, :]
    latitude = np.linspace(
        -0.5 * np.pi, 0.5 * np.pi, int(height)
    )[:, None]
    dx = np.cos(latitude) * np.cos(longitude)
    dy = np.cos(latitude) * np.sin(longitude)
    dz = np.broadcast_to(np.sin(latitude), dx.shape)

    up = np.clip((dz + 0.08) / 1.08, 0.0, 1.0)[..., None]
    down = np.clip((-dz - 0.08) / 0.92, 0.0, 1.0)[..., None]
    horizon = np.array((0.82, 0.90, 0.97))
    zenith = np.array((0.32, 0.52, 0.78))
    nadir = np.array((0.48, 0.55, 0.62))
    rgb = horizon * (1.0 - up) + zenith * up
    rgb = rgb * (1.0 - down) + nadir * down

    sun = np.array((-0.35, -0.45, 0.82))
    sun /= np.linalg.norm(sun)
    glow = np.clip(
        dx * sun[0] + dy * sun[1] + dz * sun[2], 0.0, 1.0
    )[..., None] ** 72
    rgb = np.clip(
        rgb + glow * np.array((0.42, 0.33, 0.18)), 0.0, 1.0
    )
    image = vtk_rgb_image(np.rint(rgb * 255.0))
    texture = vtk.vtkTexture()
    texture.SetInputData(image)
    texture.SetColorModeToDirectScalars()
    texture.InterpolateOn()
    texture.MipmapOn()
    texture.UseSRGBColorSpaceOff()
    return image, texture


def make_lut(cmap_name, value_range):
    lo, hi = map(float, value_range)
    table = vtk.vtkLookupTable()
    table.SetNumberOfTableValues(256)
    table.SetTableRange(lo, hi)
    cmap = colormaps[cmap_name]
    for index in range(256):
        r, g, b, a = cmap(index / 255.0)
        table.SetTableValue(index, r, g, b, a)
    table.SetNanColor(0.5, 0.5, 0.5, 0.0)
    table.Build()
    return table


def structured_grid(x, y, z, arrays, *, dynamic=False):
    """构造 VTK structured grid；i 轴最快，对应 NumPy order='F'。"""
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
    xyz = np.ascontiguousarray(
        np.column_stack((
            X.ravel(order="F"),
            Y.ravel(order="F"),
            Z.ravel(order="F"),
        )),
        dtype=np.float32,
    )
    points = vtk.vtkPoints()
    points.SetData(numpy_to_vtk(xyz, deep=True))
    grid = vtk.vtkStructuredGrid()
    grid.SetDimensions(len(x), len(y), len(z))
    grid.SetPoints(points)

    buffers, vtk_arrays = {}, {}
    for name, values in arrays.items():
        buffer = np.ascontiguousarray(
            np.asarray(values).ravel(order="F"), dtype=np.float32
        )
        vtk_array = numpy_to_vtk(buffer, deep=not dynamic)
        vtk_array.SetName(name)
        grid.GetPointData().AddArray(vtk_array)
        buffers[name] = buffer
        vtk_arrays[name] = vtk_array
    return grid, buffers, vtk_arrays


def goldak_grid(gg, arrays, *, dynamic=False, x_start=None):
    if x_start is None:
        x_start = float(cfg.run.goldak.x_start)
    gx = (gg.x - x_start + P0[0]) * 1e3
    gy = (gg.y + P0[1]) * 1e3
    gz = (P0[2] - gg.z) * 1e3
    grid, buffers, vtk_arrays = structured_grid(
        gx, gy, gz, arrays, dynamic=dynamic
    )
    return grid, gx, gy, gz, buffers, vtk_arrays


def contour_filter(grid, scalars, value):
    contour = vtk.vtkContourFilter()
    contour.SetInputData(grid)
    contour.SetInputArrayToProcess(
        0, 0, 0, vtk.vtkDataObject.FIELD_ASSOCIATION_POINTS, scalars
    )
    contour.SetValue(0, float(value))
    contour.ComputeNormalsOn()
    return contour


def slice_filter(grid, origin, normal=(0.0, 0.0, 1.0)):
    plane = vtk.vtkPlane()
    plane.SetOrigin(*map(float, origin))
    plane.SetNormal(*map(float, normal))
    cutter = vtk.vtkCutter()
    cutter.SetInputData(grid)
    cutter.SetCutFunction(plane)
    return cutter


def threshold_upper(source, scalars, value):
    threshold = vtk.vtkThreshold()
    threshold.SetInputConnection(source.GetOutputPort())
    threshold.SetInputArrayToProcess(
        0, 0, 0, vtk.vtkDataObject.FIELD_ASSOCIATION_POINTS, scalars
    )
    threshold.AllScalarsOff()  # 与 PyVista threshold(all_scalars=False) 一致
    threshold.SetUpperThreshold(float(value))
    threshold.SetThresholdFunction(vtk.vtkThreshold.THRESHOLD_UPPER)
    geometry = vtk.vtkGeometryFilter()
    geometry.SetInputConnection(threshold.GetOutputPort())
    return geometry


MOVABLE_LAYER_GROUPS = {
    "robot": "robot",
    "workpiece": "process",
    "seam": "process",
    "executed_path": "process",
    "thermal": "process",
    "domain": "process",
    "convection": "process",
    "history": "process",
    "pool": "process",
    "halo": "process",
}


def axis_rotation_matrix(axis, angle_degrees):
    """返回绕 world-space 轴旋转的齐次矩阵。"""
    axis = np.asarray(axis, dtype=float)
    norm = np.linalg.norm(axis)
    if norm < 1e-12:
        return np.eye(4)
    x, y, z = axis / norm
    angle = np.deg2rad(float(angle_degrees))
    c, s, one_c = np.cos(angle), np.sin(angle), 1.0 - np.cos(angle)
    matrix = np.eye(4)
    matrix[:3, :3] = (
        (c + x*x*one_c, x*y*one_c - z*s, x*z*one_c + y*s),
        (y*x*one_c + z*s, c + y*y*one_c, y*z*one_c - x*s),
        (z*x*one_c - y*s, z*y*one_c + x*s, c + z*z*one_c),
    )
    return matrix


class LayerTransformInteractor(vtk.vtkInteractorStyleTrackballCamera):
    """相机、对象变换和直接 TCP IK 拖动。"""

    def __init__(self, scene):
        super().__init__()
        self.scene = scene
        self.drag_group = None
        self.drag_mode = None
        self.drag_depth = 0.0
        self.drag_pivot = None
        self.drag_cursor_offset = None
        self.last_position = None
        self._observer_tags = (
            self.AddObserver(
                "LeftButtonPressEvent", self._on_left_press
            ),
            self.AddObserver(
                "LeftButtonReleaseEvent", self._on_left_release
            ),
            self.AddObserver("MouseMoveEvent", self._on_mouse_move),
        )

    def _on_left_press(self, _caller, _event):
        interactor = self.GetInteractor()
        scene = self.scene
        if interactor is None or scene is None or scene._closed:
            return
        x, y = map(float, interactor.GetEventPosition())
        target, group, pick_position = scene.pick_interaction_target(x, y)
        control = bool(interactor.GetControlKey())
        shift = bool(interactor.GetShiftKey())
        if target == "tip" and not (control or shift):
            scene.select_transform_group(None, render=False)
            tip_position = scene.tip_world_position()
            self.drag_depth = scene.world_to_display(tip_position)[2]
            cursor_world = scene.display_to_world(x, y, self.drag_depth)
            if cursor_world is not None and scene.begin_tip_drag():
                self.drag_group = group
                self.drag_mode = "tip"
                self.drag_cursor_offset = tip_position - cursor_world
                self.last_position = scene.normalize_display_position(
                    (x, y)
                )
                scene.notify_transform_started()
                interactor.Render()
                return
        scene.select_transform_group(group, render=False)
        if group is not None and (control or shift):
            self.drag_group = group
            self.drag_mode = "rotate" if control else "translate"
            self.drag_depth = scene.world_to_display(pick_position)[2]
            self.drag_pivot = scene.transform_group_center(group)
            self.last_position = scene.normalize_display_position(
                (x, y)
            )
            scene.notify_transform_started()
            interactor.Render()
            return
        self._clear_drag()
        self.OnLeftButtonDown()

    def _on_mouse_move(self, _caller, _event):
        interactor = self.GetInteractor()
        scene = self.scene
        if interactor is None or scene is None or scene._closed:
            return
        if self.drag_group is None:
            self.OnMouseMove()
            return
        position = scene.normalize_display_position(
            interactor.GetEventPosition()
        )
        if self.drag_mode == "tip":
            display_position = scene.denormalize_display_position(position)
            cursor_world = scene.display_to_world(
                *display_position, self.drag_depth
            )
            if cursor_world is not None:
                scene.drag_tip(cursor_world + self.drag_cursor_offset)
        else:
            scene.drag_transform_group(
                self.drag_group,
                self.drag_mode,
                self.last_position,
                position,
                self.drag_depth,
                self.drag_pivot,
            )
        self.last_position = position
        interactor.Render()

    def _on_left_release(self, _caller, _event):
        if self.drag_group is None:
            self.OnLeftButtonUp()
            return
        self._finish_drag()
        interactor = self.GetInteractor()
        if interactor is not None:
            interactor.Render()

    def _clear_drag(self):
        self.drag_group = None
        self.drag_mode = None
        self.drag_pivot = None
        self.drag_cursor_offset = None
        self.last_position = None

    def _finish_drag(self):
        if self.drag_mode == "tip" and self.scene is not None:
            self.scene.end_tip_drag()
        self._clear_drag()

    def detach(self):
        self._finish_drag()
        for tag in self._observer_tags:
            self.RemoveObserver(tag)
        self._observer_tags = ()
        self.scene = None


class WeldingRTXSceneBase:
    """FrameScene 基类：显式 NVIDIA EGL + actor 分组 + 相机/截图。"""

    def __init__(
        self, *, layers, frame_count, current_frame, camera_home,
        size=(960, 640),
        floor=(-650.0, 1150.0, -700.0, 700.0, -31.5, 100.0),
    ):
        if int(frame_count) <= 0:
            raise ValueError("frame_count must be positive")
        self.layers = tuple(layers)
        self.frame_count = int(frame_count)
        self.current_frame = max(
            0, min(int(current_frame), self.frame_count - 1)
        )
        self.caption = ""
        self._closed = False
        self._records = []
        self._transform_records = {}
        self._transform_poses = {}
        self._transform_objects = {}
        self._prop_transform_group = {}
        self._prop_interaction_target = {}
        self._selected_transform_group = None
        self._transform_start_callback = None
        self._tip_ik_status_callback = None
        self._interaction_style = None
        self._tip_ik = None
        self._tip_ik_enabled = False
        self._tip_drag_rotation = None
        self._tip_drag_posture_q = None
        self._tip_override_q = None
        self._tip_override_frame = None
        self._nominal_robot_q = None
        self._interaction_picker = vtk.vtkCellPicker()
        self._interaction_picker.SetTolerance(0.005)
        self._interaction_picker.PickFromListOn()
        self._layer_visible = {
            spec.key: bool(spec.visible)
            for spec in self.layers
            if spec.key != "instant_mode"
        }
        self._instant_mode = next(
            (
                bool(spec.visible)
                for spec in self.layers
                if spec.key == "instant_mode"
            ),
            False,
        )
        self._layer_opacity = {"robot": 1.0}
        self.context = NvidiaEGLContext(
            size=size,
            device_index=0,
            multisamples=8,
            background=(0.78, 0.86, 0.94),
            background2=(0.30, 0.50, 0.72),
        )
        self.renderer = self.context.renderer
        self.render_window = self.context.render_window
        try:
            self._build_environment(floor)
            self._build_scene()
            self._build_selection_outline()
            self.set_frame(self.current_frame, render=False)
            self._set_camera(camera_home)
            self._camera_base = self._capture_camera()
            self._camera_home = dict(self._camera_base)
            self._interaction_style = LayerTransformInteractor(self)
            self.context.interactor.SetInteractorStyle(
                self._interaction_style
            )
            self.backend_info = self.context.verify()
        except Exception:
            if self._interaction_style is not None:
                self._interaction_style.detach()
            self.context.close()
            raise

    def _build_environment(self, floor):
        """添加不影响相机边界的天空、有限地坪和摄影棚灯光。"""
        xmin, xmax, ymin, ymax, z_floor, spacing = map(float, floor)
        if not (xmin < xmax and ymin < ymax and spacing > 0.0):
            raise ValueError("invalid floor bounds or spacing")

        self.renderer.AutomaticLightCreationOff()
        self.renderer.RemoveAllLights()
        self.renderer.SetAmbient(0.18, 0.20, 0.24)
        self._light_kit = vtk.vtkLightKit()
        self._light_kit.MaintainLuminanceOn()
        self._light_kit.SetKeyLightIntensity(0.82)
        self._light_kit.SetKeyToFillRatio(2.5)
        self._light_kit.SetKeyToHeadRatio(4.0)
        self._light_kit.SetKeyToBackRatio(3.0)
        self._light_kit.SetKeyLightWarmth(0.58)
        self._light_kit.SetFillLightWarmth(0.42)
        self._light_kit.SetHeadLightWarmth(0.50)
        self._light_kit.SetBackLightWarmth(0.54)
        self._light_kit.AddLightsToRenderer(self.renderer)

        self._sky_image, self._sky_texture = make_sky_texture()
        sky = vtk.vtkSkybox()
        sky.SetTexture(self._sky_texture)
        sky.SetProjectionToSphere()
        sky.UseBoundsOff()
        sky.PickableOff()
        sky.DragableOff()
        self._sky_record = self._register(sky, "environment")

        floor_source = vtk.vtkPlaneSource()
        floor_source.SetOrigin(xmin, ymin, z_floor)
        floor_source.SetPoint1(xmax, ymin, z_floor)
        floor_source.SetPoint2(xmin, ymax, z_floor)
        floor_source.SetXResolution(max(1, round((xmax - xmin) / spacing)))
        floor_source.SetYResolution(max(1, round((ymax - ymin) / spacing)))
        floor_mapper = vtk.vtkPolyDataMapper()
        floor_mapper.SetInputConnection(floor_source.GetOutputPort())
        floor_actor = vtk.vtkActor()
        floor_actor.SetMapper(floor_mapper)
        floor_actor.PickableOff()
        floor_actor.DragableOff()
        floor_property = floor_actor.GetProperty()
        floor_property.SetColor(*_rgb("#74808a"))
        floor_property.EdgeVisibilityOn()
        floor_property.SetEdgeColor(*_rgb("#46535e"))
        floor_property.SetLineWidth(0.8)
        floor_property.SetInterpolationToPhong()
        floor_property.SetAmbient(0.25)
        floor_property.SetDiffuse(0.72)
        floor_property.SetSpecular(0.12)
        floor_property.SetSpecularPower(24.0)
        self._floor_record = self._register(
            floor_actor, "environment", opacity=1.0
        )

    def _build_scene(self):
        raise NotImplementedError

    def _update_frame(self, index):
        raise NotImplementedError

    def _build_selection_outline(self):
        self._selection_outline_source = vtk.vtkOutlineSource()
        self._selection_outline_source.SetBounds(-1, 1, -1, 1, -1, 1)
        mapper = vtk.vtkPolyDataMapper()
        mapper.SetInputConnection(
            self._selection_outline_source.GetOutputPort()
        )
        self._selection_outline = vtk.vtkActor()
        self._selection_outline.SetMapper(mapper)
        self._selection_outline.GetProperty().SetColor(1.0, 0.72, 0.08)
        self._selection_outline.GetProperty().SetLineWidth(3.0)
        self._selection_outline.GetProperty().LightingOff()
        self._selection_outline.PickableOff()
        self._selection_outline.DragableOff()
        self._selection_outline.UseBoundsOff()
        self._selection_outline.VisibilityOff()
        self.renderer.AddActor(self._selection_outline)

    def _ensure_transform_group(self, group):
        if group not in self._transform_objects:
            self._transform_objects[group] = vtk.vtkTransform()
            self._transform_poses[group] = np.eye(4)
            self._transform_records[group] = []
        return self._transform_objects[group]

    def _set_transform_pose(self, group, pose):
        pose = np.asarray(pose, dtype=float).reshape(4, 4)
        matrix = vtk.vtkMatrix4x4()
        for row in range(4):
            for column in range(4):
                matrix.SetElement(row, column, pose[row, column])
        self._transform_poses[group] = pose
        transform = self._transform_objects[group]
        transform.SetMatrix(matrix)
        transform.Modified()

    def transform_group_bounds(self, group):
        box = vtk.vtkBoundingBox()
        for record in self._transform_records.get(group, ()):
            prop = record["prop"]
            if not prop.GetVisibility():
                continue
            bounds = prop.GetBounds()
            if bounds is not None and np.all(np.isfinite(bounds)):
                box.AddBounds(bounds)
        if not box.IsValid():
            return None
        bounds = [0.0] * 6
        box.GetBounds(bounds)
        return np.asarray(bounds, dtype=float)

    def transform_group_center(self, group):
        bounds = self.transform_group_bounds(group)
        if bounds is None:
            return np.zeros(3)
        return np.array([
            0.5 * (bounds[0] + bounds[1]),
            0.5 * (bounds[2] + bounds[3]),
            0.5 * (bounds[4] + bounds[5]),
        ])

    def _update_selection_outline(self):
        group = self._selected_transform_group
        bounds = (
            None if group is None else self.transform_group_bounds(group)
        )
        if bounds is None:
            self._selection_outline.VisibilityOff()
            return
        diagonal = np.linalg.norm(bounds[[1, 3, 5]] - bounds[[0, 2, 4]])
        padding = max(1.0, 0.0125 * diagonal)
        padded = bounds + np.array([
            -padding, padding, -padding, padding, -padding, padding
        ])
        self._selection_outline_source.SetBounds(tuple(padded))
        self._selection_outline_source.Modified()
        self._selection_outline.VisibilityOn()

    def select_transform_group(self, group, *, render=True):
        if group not in self._transform_records:
            group = None
        self._selected_transform_group = group
        self._update_selection_outline()
        if render:
            self.render_window.Render()

    def set_prop_interaction_target(
        self, prop, target, group, *, enabled,
    ):
        """把独立交互 prop 加入或移出 picker。"""
        address = prop.GetAddressAsString("")
        if enabled:
            if address not in self._prop_interaction_target:
                self._interaction_picker.AddPickList(prop)
            self._prop_interaction_target[address] = str(target)
            self._prop_transform_group[address] = group
            prop.PickableOn()
            return
        if address in self._prop_interaction_target:
            self._interaction_picker.DeletePickList(prop)
        self._prop_interaction_target.pop(address, None)
        self._prop_transform_group.pop(address, None)
        prop.PickableOff()

    def pick_interaction_target(self, x, y):
        """返回 (交互目标, 变换组, world 坐标)。"""
        if self._closed:
            return None, None, None
        if self.tip_handle_hit_test(x, y):
            return "tip", "robot", self.tip_world_position()
        picked = self._interaction_picker.Pick(
            float(x), float(y), 0.0, self.renderer
        )
        prop = self._interaction_picker.GetViewProp() if picked else None
        if prop is None:
            return None, None, None
        address = prop.GetAddressAsString("")
        group = self._prop_transform_group.get(address)
        if group is None:
            return None, None, None
        target = self._prop_interaction_target.get(address, "transform")
        position = np.asarray(
            self._interaction_picker.GetPickPosition(), dtype=float
        )
        return target, group, position

    def tip_handle_hit_test(self, x, y):
        """给可见 TCP 手柄稳定的屏幕空间优先命中区域。"""
        if not self._tip_ik_enabled or not hasattr(self, "robot"):
            return False
        prop = self.robot.tip_handle_record["prop"]
        if not prop.GetVisibility() or not prop.GetPickable():
            return False
        if self.robot.tip_position_mm is None:
            return False
        center = self.world_to_display(self.tip_world_position())
        if not np.all(np.isfinite(center[:2])):
            return False
        return bool(
            np.hypot(float(x) - center[0], float(y) - center[1])
            <= TIP_HANDLE_HIT_RADIUS_PX
        )

    def pick_transform_group(self, x, y):
        """只关心刚体组的兼容接口。"""
        _, group, position = self.pick_interaction_target(x, y)
        return group, position

    def world_to_display(self, point):
        self.renderer.SetWorldPoint(*map(float, point), 1.0)
        self.renderer.WorldToDisplay()
        return np.asarray(self.renderer.GetDisplayPoint(), dtype=float)

    def display_to_world(self, x, y, depth):
        self.renderer.SetDisplayPoint(float(x), float(y), float(depth))
        self.renderer.DisplayToWorld()
        point = np.asarray(self.renderer.GetWorldPoint(), dtype=float)
        if abs(point[3]) < 1e-12:
            return None
        return point[:3] / point[3]

    def normalize_display_position(self, position):
        """把 framebuffer 像素归一化，避免交互降采样时拖动跳变。"""
        width, height = self.render_window.GetSize()
        return (
            float(position[0]) / max(1.0, float(width)),
            float(position[1]) / max(1.0, float(height)),
        )

    def denormalize_display_position(self, position):
        width, height = self.render_window.GetSize()
        return (
            float(position[0]) * max(1.0, float(width)),
            float(position[1]) * max(1.0, float(height)),
        )

    def notify_transform_started(self):
        if self._transform_start_callback is not None:
            self._transform_start_callback()

    def set_transform_start_callback(self, callback):
        self._transform_start_callback = callback

    def set_tip_ik_status_callback(self, callback):
        self._tip_ik_status_callback = callback

    def _emit_tip_ik_status(self, level, message):
        if self._tip_ik_status_callback is not None:
            self._tip_ik_status_callback(str(level), str(message))

    def enable_tip_ik(self, enabled=True, *, render=True):
        """启用本 RTX 场景的直接 TCP Mink IK 拖动。"""
        enabled = bool(enabled)
        if enabled == self._tip_ik_enabled:
            return
        if enabled:
            if not hasattr(self, "robot"):
                raise TypeError("tip IK requires a robot scene")
            self._tip_ik = MinkArmIK(self.robot.arm)
            self._tip_ik_enabled = True
            self.robot.set_tip_handle_enabled(True)
            self._emit_tip_ik_status(
                "ready", "Mink IK ready — drag the orange TCP handle"
            )
        else:
            self.end_tip_drag()
            self.clear_tip_ik_override(render=False, notify=False)
            if hasattr(self, "robot"):
                self.robot.set_tip_handle_enabled(False)
            self._tip_ik_enabled = False
            self._tip_ik = None
        self._apply_visibility()
        self.renderer.ResetCameraClippingRange()
        if render:
            self.render_window.Render()

    def tip_world_position(self):
        """返回叠加 robot 刚体变换后的 TCP 坐标 [mm]。"""
        if not hasattr(self, "robot") or self.robot.tip_position_mm is None:
            raise RuntimeError("robot pose has not been initialized")
        local = np.append(self.robot.tip_position_mm, 1.0)
        pose = self._transform_poses.get("robot", np.eye(4))
        return (pose @ local)[:3]

    def _robot_local_position(self, world_position):
        pose = self._transform_poses.get("robot", np.eye(4))
        local = np.linalg.solve(
            pose, np.append(np.asarray(world_position, dtype=float), 1.0)
        )
        if abs(local[3]) < 1e-12:
            raise ValueError(
                "robot transform produced an invalid homogeneous position"
            )
        return local[:3] / local[3]

    def begin_tip_drag(self):
        """捕获一次 IK 拖动的工具姿态和正则构型。"""
        if not self._tip_ik_enabled or self._tip_ik is None:
            return False
        q = self.robot.current_q
        if q is None:
            return False
        self._tip_drag_posture_q = np.asarray(q, dtype=float).copy()
        self._tip_drag_rotation = np.asarray(
            self.robot.tip_rotation, dtype=float
        ).copy()
        self.robot.set_tip_handle_error(False)
        self._emit_tip_ik_status("active", "Mink IK solving…")
        return True

    def drag_tip(self, target_world_mm):
        """求解并显示一个保持工具姿态的 TCP 鼠标目标。"""
        if (
            not self._tip_ik_enabled
            or self._tip_ik is None
            or self._tip_drag_rotation is None
        ):
            return False
        q_seed = np.asarray(self.robot.current_q, dtype=float)
        try:
            target_position_m = (
                self._robot_local_position(target_world_mm) * 1e-3
            )
            result = self._tip_ik.solve(
                q_seed,
                target_position_m,
                self._tip_drag_rotation,
                posture_q=self._tip_drag_posture_q,
            )
            q = np.asarray(result.q, dtype=float)
            achieved = np.asarray(result.achieved_position_m, dtype=float)
            position_error_m = float(result.position_error_m)
            orientation_error_rad = float(result.orientation_error_rad)
            finite = (
                q.shape == (6,)
                and achieved.shape == (3,)
                and np.all(np.isfinite(q))
                and np.all(np.isfinite(achieved))
                and np.isfinite(position_error_m)
                and np.isfinite(orientation_error_rad)
            )
            if not finite:
                raise ValueError("Mink returned a non-finite IK result")
        except Exception as exc:
            self.robot.set_tip_handle_error(True)
            self._emit_tip_ik_status(
                "error", f"Mink IK error: {type(exc).__name__}: {exc}"
            )
            return False

        self._tip_override_q = q.copy()
        self._tip_override_frame = self.current_frame
        self.robot.update(q)
        converged = bool(result.converged)
        self.robot.set_tip_handle_error(not converged)
        level = "ok" if converged else "error"
        label = "converged" if converged else "target not converged"
        self._emit_tip_ik_status(
            level,
            f"Mink IK {label} — Δp {position_error_m * 1e3:.4f} mm · "
            f"ΔR {orientation_error_rad:.5f} rad",
        )
        self.renderer.ResetCameraClippingRange()
        return converged

    def end_tip_drag(self):
        self._tip_drag_rotation = None
        self._tip_drag_posture_q = None

    def _update_robot_pose(self, q):
        """保存记录帧构型并应用当前帧的 IK 覆盖。"""
        self._nominal_robot_q = np.asarray(q, dtype=float).copy()
        display_q = self._nominal_robot_q
        if (
            self._tip_override_q is not None
            and self._tip_override_frame == self.current_frame
        ):
            display_q = self._tip_override_q
        self.robot.update(display_q)

    def clear_tip_ik_override(self, *, render=True, notify=True):
        """恢复当前帧的记录构型。"""
        had_override = self._tip_override_q is not None
        self.end_tip_drag()
        self._tip_override_q = None
        self._tip_override_frame = None
        if self._nominal_robot_q is not None and hasattr(self, "robot"):
            self.robot.update(self._nominal_robot_q)
        if hasattr(self, "robot"):
            self.robot.set_tip_handle_error(False)
        if notify and self._tip_ik_enabled and had_override:
            self._emit_tip_ik_status(
                "ready", "Recorded frame pose restored"
            )
        self.renderer.ResetCameraClippingRange()
        if render:
            self.render_window.Render()

    def _apply_transform_delta(self, group, delta):
        pose = np.asarray(delta, dtype=float) @ self._transform_poses[group]
        self._set_transform_pose(group, pose)
        self._update_selection_outline()
        self.renderer.ResetCameraClippingRange()

    def drag_transform_group(
        self, group, mode, previous, current, depth, pivot,
    ):
        if group not in self._transform_poses or previous == current:
            return
        # VtkRemoteView 的 interactive_ratio 会在拖动期间调整 framebuffer；
        # 用同一个当前尺寸还原两端，避免 resize 被误当成鼠标位移。
        previous = self.denormalize_display_position(previous)
        current = self.denormalize_display_position(current)
        if mode == "translate":
            before = self.display_to_world(*previous, depth)
            after = self.display_to_world(*current, depth)
            if before is None or after is None:
                return
            delta = np.eye(4)
            delta[:3, 3] = after - before
        elif mode == "rotate":
            dx = float(current[0] - previous[0])
            dy = float(current[1] - previous[1])
            width, height = self.render_window.GetSize()
            sensitivity = 180.0 / max(1.0, min(width, height))
            camera = self.renderer.GetActiveCamera()
            view_up = np.asarray(camera.GetViewUp(), dtype=float)
            direction = np.asarray(
                camera.GetDirectionOfProjection(), dtype=float
            )
            camera_right = np.cross(direction, view_up)
            to_origin = np.eye(4)
            to_origin[:3, 3] = -np.asarray(pivot, dtype=float)
            from_origin = np.eye(4)
            from_origin[:3, 3] = np.asarray(pivot, dtype=float)
            yaw = axis_rotation_matrix(view_up, -dx * sensitivity)
            pitch = axis_rotation_matrix(camera_right, dy * sensitivity)
            delta = from_origin @ yaw @ pitch @ to_origin
        else:
            raise ValueError(f"unknown drag mode: {mode}")
        self._apply_transform_delta(group, delta)

    def reset_object_transforms(self, *, render=True):
        for group in tuple(self._transform_poses):
            self._set_transform_pose(group, np.eye(4))
        self.clear_tip_ik_override(render=False)
        self.select_transform_group(None, render=False)
        self.renderer.ResetCameraClippingRange()
        if render:
            self.render_window.Render()

    def _register(
        self, prop, layer, *, mode=None, opacity=None, enabled=True,
        transform_group="auto", pickable=True,
    ):
        group = (
            MOVABLE_LAYER_GROUPS.get(layer)
            if transform_group == "auto" else transform_group
        )
        movable = group is not None and isinstance(prop, vtk.vtkProp3D)
        record = {
            "prop": prop,
            "layer": layer,
            "mode": mode,
            "opacity": opacity,
            "enabled": bool(enabled),
            "transform_group": group if movable else None,
        }
        if movable:
            transform = self._ensure_transform_group(group)
            prop.SetUserTransform(transform)
            self._transform_records[group].append(record)
            if pickable:
                prop.PickableOn()
                self._interaction_picker.AddPickList(prop)
                self._prop_transform_group[
                    prop.GetAddressAsString("")
                ] = group
            else:
                prop.PickableOff()
        else:
            prop.PickableOff()
        self._records.append(record)
        self.renderer.AddViewProp(prop)
        self._apply_record(record)
        return record

    def _apply_record(self, record):
        mode = record["mode"]
        mode_visible = (
            mode is None
            or (mode == "instant" and self._instant_mode)
            or (mode == "peak" and not self._instant_mode)
        )
        factor = self._layer_opacity.get(record["layer"], 1.0)
        visible = (
            record["enabled"]
            and self._layer_visible.get(record["layer"], True)
            and mode_visible
            and factor > 0.02
        )
        record["prop"].SetVisibility(bool(visible))
        if record["opacity"] is not None:
            record["prop"].GetProperty().SetOpacity(
                float(record["opacity"]) * factor
            )

    def _apply_visibility(self):
        for record in self._records:
            self._apply_record(record)

    def set_record_enabled(self, record, enabled):
        record["enabled"] = bool(enabled)
        self._apply_record(record)

    def add_surface(
        self, source, layer, *, mode=None, color="#a7a9ac",
        opacity=1.0, scalars=None, cmap="viridis", clim=None,
        smooth=True, transform_group="auto", pickable=True,
    ):
        mapper = vtk.vtkDataSetMapper()
        if hasattr(source, "GetOutputPort"):
            mapper.SetInputConnection(source.GetOutputPort())
        else:
            mapper.SetInputData(source)
        lut = None
        if scalars is None:
            mapper.ScalarVisibilityOff()
        else:
            mapper.ScalarVisibilityOn()
            mapper.SetScalarModeToUsePointFieldData()
            mapper.SelectColorArray(scalars)
            mapper.SetColorModeToMapScalars()
            if clim is None:
                raise ValueError("clim is required for scalar mapping")
            lut = make_lut(cmap, clim)
            mapper.SetLookupTable(lut)
            mapper.SetScalarRange(*map(float, clim))
            mapper.UseLookupTableScalarRangeOn()
            mapper.InterpolateScalarsBeforeMappingOn()

        actor = vtk.vtkActor()
        actor.SetMapper(mapper)
        actor.GetProperty().SetColor(*_rgb(color))
        actor.GetProperty().SetInterpolationToPhong()
        thermal_layer = scalars is not None or layer in {
            "thermal", "convection", "history", "pool", "halo"
        }
        actor.GetProperty().SetAmbient(0.50 if thermal_layer else 0.24)
        actor.GetProperty().SetDiffuse(0.50 if thermal_layer else 0.76)
        actor.GetProperty().SetSpecular(
            0.0 if thermal_layer or not smooth else 0.16
        )
        actor.GetProperty().SetSpecularPower(24.0)
        record = self._register(
            actor, layer, mode=mode, opacity=float(opacity),
            transform_group=transform_group, pickable=pickable,
        )
        return record, lut

    def add_scalar_bar(
        self, lut, title, layer, *, mode=None, position=(0.83, 0.08)
    ):
        bar = vtk.vtkScalarBarActor()
        bar.SetLookupTable(lut)
        bar.SetTitle(title)
        bar.SetNumberOfLabels(5)
        bar.SetPosition(*position)
        bar.SetWidth(0.14)
        bar.SetHeight(0.34)
        bar.DrawBackgroundOn()
        bar.GetBackgroundProperty().SetColor(0.08, 0.12, 0.18)
        bar.GetBackgroundProperty().SetOpacity(0.22)
        bar.GetTitleTextProperty().SetColor(0.96, 0.97, 0.99)
        bar.GetLabelTextProperty().SetColor(0.96, 0.97, 0.99)
        return self._register(bar, layer, mode=mode)

    def add_text(
        self, text, layer="annotation", *, position=(12, 12),
        font_size=18, color="#f4f7fa",
    ):
        actor = vtk.vtkTextActor()
        actor.SetInput(str(text))
        actor.SetDisplayPosition(*map(int, position))
        actor.GetTextProperty().SetFontSize(int(font_size))
        actor.GetTextProperty().SetColor(*_rgb(color))
        actor.GetTextProperty().SetFontFamilyToArial()
        actor.GetTextProperty().ShadowOn()
        return self._register(actor, layer)

    def add_axes(
        self, layer="axes", origin=(0, 0, 0), length=100.0, *,
        transform_group=None,
    ):
        """添加世界坐标轴；3-D 标签与轴保持比例并随相机缩放。"""
        length = float(length)
        ox, oy, oz = map(float, origin)
        axis_specs = (
            ("x [mm]", np.array((1.0, 0.0, 0.0)), "#a31621"),
            ("y [mm]", np.array((0.0, 1.0, 0.0)), "#16833a"),
            ("z [mm]", np.array((0.0, 0.0, 1.0)), "#1f4fb2"),
        )
        axes_record = None
        for _, direction, color in axis_specs:
            # vtkAxesActor 的 Prop3D 位移不会反映在其复合几何/边界中，
            # 非原点坐标轴会留在 world origin。把每根箭头烘焙到世界坐标，
            # 使渲染、clipping range 与后续逐帧 view_update 使用同一边界。
            helper = np.array(
                (0.0, 0.0, 1.0)
                if abs(direction[2]) < 0.9
                else (0.0, 1.0, 0.0)
            )
            axis_y = np.cross(helper, direction)
            axis_y /= np.linalg.norm(axis_y)
            axis_z = np.cross(direction, axis_y)
            basis = np.column_stack((direction, axis_y, axis_z))
            matrix = vtk.vtkMatrix4x4()
            for row in range(3):
                for column in range(3):
                    matrix.SetElement(
                        row, column, length * basis[row, column]
                    )
            matrix.SetElement(0, 3, ox)
            matrix.SetElement(1, 3, oy)
            matrix.SetElement(2, 3, oz)
            transform = vtk.vtkTransform()
            transform.SetMatrix(matrix)
            arrow = vtk.vtkArrowSource()
            arrow.SetShaftRadius(0.025)
            arrow.SetShaftResolution(24)
            arrow.SetTipRadius(0.09)
            arrow.SetTipLength(0.25)
            arrow.SetTipResolution(32)
            placed = vtk.vtkTransformPolyDataFilter()
            placed.SetInputConnection(arrow.GetOutputPort())
            placed.SetTransform(transform)
            record, _ = self.add_surface(
                placed, layer, color=color, opacity=1.0,
                transform_group=transform_group, pickable=False,
            )
            record["prop"].PickableOff()
            if axes_record is None:
                axes_record = record

        # vtkAxesActor 的 vtkCaptionActor2D 标签按 viewport 缩放，模型
        # 拉远后仍占据大量像素。改用世界坐标 vtkVectorText；vtkFollower
        # 只负责朝向相机，几何尺寸仍随 perspective/parallel zoom 变化。
        label_offset = 1.25 * length
        for text, direction, color in axis_specs:
            text_source = vtk.vtkVectorText()
            text_source.SetText(text)
            text_source.Update()
            xmin, xmax, ymin, ymax, _, _ = text_source.GetOutput().GetBounds()
            mapper = vtk.vtkPolyDataMapper()
            mapper.SetInputConnection(text_source.GetOutputPort())
            mapper.ScalarVisibilityOff()
            label = vtk.vtkFollower()
            label.SetMapper(mapper)
            cx = 0.5 * (xmin + xmax)
            cy = 0.5 * (ymin + ymax)
            label_size = 0.10 * length / max(ymax - ymin, 1e-9)
            label.SetOrigin(cx, cy, 0.0)
            label.SetPosition(
                ox + direction[0] * label_offset - cx,
                oy + direction[1] * label_offset - cy,
                oz + direction[2] * label_offset,
            )
            label.SetScale(label_size, label_size, label_size)
            label.SetCamera(self.renderer.GetActiveCamera())
            label.GetProperty().SetColor(*_rgb(color))
            label.GetProperty().LightingOff()
            label.PickableOff()
            label.DragableOff()
            self._register(
                label, layer, transform_group=transform_group,
                pickable=False,
            )
        return axes_record

    def set_frame(self, index, *, render=True):
        if self._closed:
            return self.caption
        index = max(0, min(int(index), self.frame_count - 1))
        if index != self.current_frame:
            self.clear_tip_ik_override(render=False)
        self.current_frame = index
        self._update_frame(index)
        self._apply_visibility()
        self._update_selection_outline()
        self.renderer.ResetCameraClippingRange()
        if render:
            self.render_window.Render()
        return self.caption

    def set_layer_visible(self, key, visible, *, render=True):
        if self._closed:
            return
        if key == "instant_mode":
            self._instant_mode = bool(visible)
        else:
            self._layer_visible[key] = bool(visible)
        self._on_layer_changed(key, bool(visible))
        self._apply_visibility()
        if key == "ik_handle" and hasattr(self, "robot"):
            handle = self.robot.tip_handle_record["prop"]
            pickable = (
                bool(visible)
                and self.robot.tip_handle_enabled
                and bool(handle.GetVisibility())
            )
            handle.SetPickable(pickable)
        self._update_selection_outline()
        self.renderer.ResetCameraClippingRange()
        if render:
            self.render_window.Render()

    def _on_layer_changed(self, key, visible):
        pass

    def set_layer_opacity(self, key, opacity, *, render=True):
        self._layer_opacity[key] = max(0.0, min(float(opacity), 1.0))
        self._apply_visibility()
        if render:
            self.render_window.Render()

    def _capture_camera(self):
        camera = self.renderer.GetActiveCamera()
        return {
            "position": camera.GetPosition(),
            "focal_point": camera.GetFocalPoint(),
            "view_up": camera.GetViewUp(),
            "view_angle": camera.GetViewAngle(),
            "parallel_projection": camera.GetParallelProjection(),
            "parallel_scale": camera.GetParallelScale(),
        }

    def _restore_camera(self, state):
        camera = self.renderer.GetActiveCamera()
        camera.SetPosition(*state["position"])
        camera.SetFocalPoint(*state["focal_point"])
        camera.SetViewUp(*state["view_up"])
        camera.SetViewAngle(float(state["view_angle"]))
        camera.SetParallelProjection(bool(state["parallel_projection"]))
        camera.SetParallelScale(float(state["parallel_scale"]))
        camera.OrthogonalizeViewUp()

    def _set_camera(self, camera_home):
        position, focal_point, view_up = camera_home
        camera = self.renderer.GetActiveCamera()
        camera.SetPosition(*map(float, position))
        camera.SetFocalPoint(*map(float, focal_point))
        camera.SetViewUp(*map(float, view_up))
        camera.SetViewAngle(30.0)
        camera.ParallelProjectionOff()
        camera.OrthogonalizeViewUp()
        self.renderer.ResetCameraClippingRange()

    def apply_view(
        self, px=0.0, py=0.0, pz=0.0, az=0.0, el=0.0,
        roll=0.0, arm_t=0.0, *, render=True,
    ):
        """从 home 相机起算绝对平移/旋转，同时设置机械臂透明度。"""
        self._restore_camera(self._camera_base)
        camera = self.renderer.GetActiveCamera()
        pan = np.array([px, py, pz], dtype=float)
        camera.SetFocalPoint(
            *(np.asarray(camera.GetFocalPoint()) + pan)
        )
        camera.SetPosition(*(np.asarray(camera.GetPosition()) + pan))
        if az:
            camera.Azimuth(float(az))
            camera.OrthogonalizeViewUp()
        if el:
            camera.Elevation(float(el))
            camera.OrthogonalizeViewUp()
        if roll:
            camera.Roll(float(roll))
            camera.OrthogonalizeViewUp()
        self.set_layer_opacity(
            "robot", 1.0 - float(arm_t) / 100.0, render=False
        )
        self.renderer.ResetCameraClippingRange()
        self._camera_home = self._capture_camera()
        if render:
            self.render_window.Render()

    def reset_camera(self, *, render=True):
        if self._closed:
            return
        self._restore_camera(self._camera_home)
        self.renderer.ResetCameraClippingRange()
        if render:
            self.render_window.Render()

    def capture_rgb(self):
        """读取 EGL render window 的后缓冲，返回 Pillow 可用 RGB 数组。"""
        self.render_window.Render()
        capture = vtk.vtkWindowToImageFilter()
        capture.SetInput(self.render_window)
        capture.SetInputBufferTypeToRGB()
        capture.ReadFrontBufferOff()
        capture.Update()
        image = capture.GetOutput()
        width, height, _ = image.GetDimensions()
        rgb = vtk_to_numpy(
            image.GetPointData().GetScalars()
        ).reshape(height, width, -1)
        return np.flipud(rgb[:, :, :3]).copy()

    def close(self):
        if self._closed:
            return
        if self._interaction_style is not None:
            self._interaction_style.detach()
            self._interaction_style = None
        self.set_tip_ik_status_callback(None)
        self.enable_tip_ik(False, render=False)
        self._closed = True
        self._transform_start_callback = None
        self._interaction_picker.InitializePickList()
        self._prop_interaction_target.clear()
        self._prop_transform_group.clear()
        self.renderer.RemoveAllViewProps()
        self.context.close()


class RobotRig:
    """持久机械臂 actor；换帧只更新 source/actor，不重建 renderer。"""

    def __init__(self, scene, arm_model, layer="robot"):
        self.scene = scene
        self.arm = arm_model
        self.links = []
        self.joints = []
        self.current_q = None
        self.tip_position_mm = None
        self.tip_rotation = None
        self.tip_handle_enabled = False

        # 底座：z=-30..0 mm 的粗圆柱。
        base_line = vtk.vtkLineSource()
        base_line.SetPoint1(0.0, 0.0, -30.0)
        base_line.SetPoint2(0.0, 0.0, 0.0)
        base_tube = vtk.vtkTubeFilter()
        base_tube.SetInputConnection(base_line.GetOutputPort())
        base_tube.SetRadius(90.0)
        base_tube.SetNumberOfSides(40)
        base_tube.CappingOn()
        scene.add_surface(
            base_tube, layer, color="#6b7078", opacity=1.0
        )

        for radius in R_LINK:
            line = vtk.vtkLineSource()
            tube = vtk.vtkTubeFilter()
            tube.SetInputConnection(line.GetOutputPort())
            tube.SetRadius(radius)
            tube.SetNumberOfSides(28)
            tube.CappingOn()
            record, _ = scene.add_surface(
                tube, layer, color="#a7a9ac", opacity=1.0
            )
            self.links.append((line, record))

        for radius in R_JOINT:
            sphere = vtk.vtkSphereSource()
            sphere.SetRadius(radius)
            sphere.SetThetaResolution(28)
            sphere.SetPhiResolution(20)
            record, _ = scene.add_surface(
                sphere, layer, color="#57a7c6", opacity=1.0
            )
            self.joints.append((sphere, record))

        self.gun = vtk.vtkConeSource()
        self.gun.SetHeight(55.0)
        self.gun.SetRadius(11.0)
        self.gun.SetResolution(32)
        self.gun_record, _ = scene.add_surface(
            self.gun, layer, color="#c0392b", opacity=1.0
        )

        self.tip_handle = vtk.vtkSphereSource()
        self.tip_handle.SetRadius(16.0)
        self.tip_handle.SetThetaResolution(32)
        self.tip_handle.SetPhiResolution(24)
        self.tip_handle_record, _ = scene.add_surface(
            self.tip_handle,
            "ik_handle",
            color="#f59e0b",
            opacity=0.96,
            transform_group="robot",
            pickable=False,
        )
        self.tip_handle_record["prop"].GetProperty().SetAmbient(0.55)
        scene.set_record_enabled(self.tip_handle_record, False)

    def set_tip_handle_enabled(self, enabled):
        self.tip_handle_enabled = bool(enabled)
        self.scene.set_record_enabled(
            self.tip_handle_record, self.tip_handle_enabled
        )
        prop = self.tip_handle_record["prop"]
        self.scene.set_prop_interaction_target(
            prop, "tip", "robot", enabled=self.tip_handle_enabled
        )
        prop.SetPickable(
            self.tip_handle_enabled and bool(prop.GetVisibility())
        )
        self.set_tip_handle_error(False)

    def set_tip_handle_error(self, failed):
        color = "#d1242f" if failed else "#f59e0b"
        self.tip_handle_record["prop"].GetProperty().SetColor(
            *_rgb(color)
        )

    def update(self, q):
        origins, _, rotation = self.arm._kin(q)
        self.current_q = np.asarray(q, dtype=float).copy()
        self.tip_position_mm = np.asarray(origins[6] * 1e3, dtype=float)
        self.tip_rotation = np.asarray(rotation, dtype=float).copy()
        origins = origins * 1e3
        tool_z = rotation[:, 2]
        ends = origins.copy()
        ends[6] = origins[6] - tool_z * 50.0
        for index, (line, record) in enumerate(self.links, start=1):
            segment = ends[index] - origins[index - 1]
            enabled = np.linalg.norm(segment) > 1e-3
            if enabled:
                line.SetPoint1(*origins[index - 1])
                line.SetPoint2(*ends[index])
            self.scene.set_record_enabled(record, enabled)
        for index, (_, record) in enumerate(self.joints, start=1):
            record["prop"].SetPosition(*origins[index])
        self.gun.SetCenter(*(origins[6] - tool_z * 27.5))
        self.gun.SetDirection(*tool_z)
        self.tip_handle.SetCenter(*origins[6])


class PolylineRig:
    """可增长折线 + tube；points/cells 在原 vtkPolyData 上更新。"""

    def __init__(self, scene, layer, color, radius, opacity=1.0):
        self.scene = scene
        self.poly = vtk.vtkPolyData()
        self.points = vtk.vtkPoints()
        self.lines = vtk.vtkCellArray()
        self.poly.SetPoints(self.points)
        self.poly.SetLines(self.lines)
        self.tube = vtk.vtkTubeFilter()
        self.tube.SetInputData(self.poly)
        self.tube.SetRadius(float(radius))
        self.tube.SetNumberOfSides(20)
        self.tube.CappingOn()
        self.record, _ = scene.add_surface(
            self.tube, layer, color=color, opacity=opacity
        )

    def update(self, points):
        points = np.ascontiguousarray(points, dtype=np.float32)
        enabled = len(points) >= 2
        if enabled:
            self.points.SetData(numpy_to_vtk(points, deep=True))
            self.lines.Reset()
            self.lines.InsertNextCell(len(points))
            for index in range(len(points)):
                self.lines.InsertCellPoint(index)
            self.points.Modified()
            self.lines.Modified()
            self.poly.Modified()
        self.scene.set_record_enabled(self.record, enabled)


def add_workpiece(scene, layer="workpiece", opacity=0.55):
    cube = vtk.vtkCubeSource()
    cube.SetBounds(
        P0[0] * 1e3 - 90, P0[0] * 1e3 + 110,
        -70, 70,
        P0[2] * 1e3 - 20, P0[2] * 1e3,
    )
    return scene.add_surface(
        cube, layer, color="#d8d2c4", opacity=opacity
    )[0]


def seam_points():
    return np.array(
        [P0 * 1e3, (P0 + [V_WELD * T_TRACK, 0, 0]) * 1e3],
        dtype=float,
    ) + [0, 0, 0.3]


class RobotRTXScene(WeldingRTXSceneBase):
    """§2/§3：静态或多帧机械臂 + 增长轨迹 + 可选末时刻热场。"""

    def __init__(
        self, frame_times, *, layers, current_frame=0,
        grow_trace=True, pool_model=None, size=(950, 620),
    ):
        self.frame_times = np.asarray(frame_times, dtype=float)
        self.grow_trace = bool(grow_trace)
        self.pool_model = pool_model
        self._pool_grid = None
        self._pool_buffers = {}
        self._pool_arrays = {}
        super().__init__(
            layers=layers,
            frame_count=len(self.frame_times),
            current_frame=current_frame,
            camera_home=(
                (960.0, -780.0, 710.0),
                (250.0, 0.0, 145.0),
                (0.0, 0.0, 1.0),
            ),
            size=size,
        )

    def _build_scene(self):
        self.robot = RobotRig(self, arm)
        add_workpiece(self)
        self.seam = PolylineRig(
            self, "seam", "#151515", radius=0.9
        )
        self.seam.update(seam_points())
        self.trace = PolylineRig(
            self, "executed_path", "#d81b1b", radius=0.6
        )
        self.add_axes(origin=(0, 0, 0), length=110.0)

        if self.pool_model is not None:
            self.attach_pool(self.pool_model, render=False)

    def attach_pool(self, pool_model, *, render=True):
        """§4 完成后把 g 原位接入已显示的 §2/§3 RTX 场景。"""
        if not all(
            hasattr(pool_model, key)
            for key in ("peak", "T", "Tm", "x", "y", "z")
        ):
            raise ValueError("pool_model 必须是已经完成 run() 的 Goldak 场")
        values = {"peak": pool_model.peak, "inst": pool_model.T}
        if self._pool_grid is None:
            (
                self._pool_grid, _, _, _, self._pool_buffers,
                self._pool_arrays,
            ) = goldak_grid(pool_model, values, dynamic=True)
            for field, mode, color, opacity in (
                ("peak", "peak", "#9f2f24", 0.88),
                ("inst", "instant", "#e02f20", 0.92),
            ):
                self.add_surface(
                    contour_filter(
                        self._pool_grid, field, float(pool_model.Tm)
                    ),
                    "thermal", mode=mode, color=color, opacity=opacity,
                )
        else:
            for field, value in values.items():
                flat = np.asarray(value).ravel(order="F")
                if flat.size != self._pool_buffers[field].size:
                    raise ValueError(
                        "热场网格尺寸已变化；请重跑 §2/§3 以重建场景"
                    )
                np.copyto(
                    self._pool_buffers[field], flat, casting="unsafe"
                )
                self._pool_arrays[field].Modified()
            self._pool_grid.GetPointData().Modified()
            self._pool_grid.Modified()
        self.pool_model = pool_model
        self._update_frame(self.current_frame)
        self._apply_visibility()
        if render:
            self.render_window.Render()

    def _update_frame(self, index):
        t_now = float(self.frame_times[index])
        self._update_robot_pose(q_at(t_now))
        if self.grow_trace:
            mask = t_tr <= t_now
            points = tip[mask] * 1e3 + [0, 0, 0.3]
        else:
            points = tip * 1e3 + [0, 0, 0.3]
        self.trace.update(points)
        suffix = "" if self._pool_grid is not None else " · run §4 for thermal field"
        self.caption = f"t = {t_now:.2f} s{suffix}"


def bind_live_controls(app, scene, scene_name, layers, *,
                       lim=300.0, step=10.0):
    controls = view_widgets(scene_name, lim=lim, step=step)
    app._external_controls = tuple(controls)

    def apply_view(_change=None):
        if app._closed:
            return
        values = dict(zip(VIEW_KEYS, (w.value for w in controls)))
        scene.apply_view(**values, render=False)
        app.ctrl.view_update()

    for control in controls:
        control.observe(apply_view, names="value")
    app._external_view_handler = apply_view
    apply_view()

    # RTX 内置帧滑块也写回原状态文件，重跑后从上次时刻继续。
    if hasattr(scene, "frame_times") and len(scene.frame_times) > 1:
        def remember_frame(frame_idx=0, **_):
            save_state(
                scene_name, "t",
                float(scene.frame_times[int(frame_idx)]),
            )

        app.state.change("frame_idx")(remember_frame)
        app._callbacks.append(("frame_idx", remember_frame))

    # 图层菜单状态持久化；inst/conv 兼容原 notebook 的键名。
    for index, spec in enumerate(layers):
        storage = {
            "instant_mode": "inst",
            "convection": "conv",
        }.get(spec.key, f"layer_{spec.key}")
        state_key = f"show_layer_{index}"

        def remember_layer(
            _storage=storage, _state_key=state_key, **values
        ):
            save_state(
                scene_name, _storage,
                bool(values.get(_state_key)),
            )

        app.state.change(state_key)(remember_layer)
        app._callbacks.append((state_key, remember_layer))

    reset_objects = widgets.Button(
        description="重置对象位姿",
        icon="undo",
        tooltip="Reset robot and process-object transforms",
        layout=widgets.Layout(width="180px"),
    )
    interaction_help = widgets.HTML(
        value=(
            "<span style='color:#4b5563'>"
            "点击选择（黄框） · Shift+左拖平移 · "
            "Ctrl+左拖旋转"
            + (
                " · TCP: 左拖橙色手柄做 Mink IK"
                if app._tip_ik_enabled else ""
            )
            + "</span>"
        ),
    )

    def reset_object_pose(_button):
        if app._closed:
            return
        with app.state:
            app.state.playing = False
        scene.reset_object_transforms(render=False)
        app.ctrl.view_update()

    reset_objects.on_click(reset_object_pose)
    app._object_reset_button = reset_objects
    app._object_reset_handler = reset_object_pose
    app._interaction_help = interaction_help

    rows = [
        widgets.HBox(controls[:3]),
        widgets.HBox(controls[3:]),
        widgets.HBox(
            (reset_objects, interaction_help),
            layout=widgets.Layout(align_items="center"),
        ),
    ]
    if app._tip_ik_enabled:
        rows.append(app._tip_ik_status)
    rows = tuple(rows)
    panel = widgets.VBox(rows)
    app._external_control_rows = rows
    app._external_control_panel = panel
    return panel


class WeldingRTXLiveWidget(RTXLiveWidget):
    """刷新图层时同步 caption；其余前端行为全部继承 vendor widget。"""

    def _pause_for_object_drag(self):
        if self._closed:
            return
        with self.state:
            self.state.playing = False

    def _set_tip_ik_status(self, level, message):
        colors = {
            "ready": "#57606a",
            "active": "#9a6700",
            "ok": "#1a7f37",
            "error": "#cf222e",
        }
        color = colors.get(str(level), colors["ready"])
        self._tip_ik_status.value = (
            f"<span style='color:{color}'>"
            f"{escape(str(message))}</span>"
        )

    async def _toggle_play(self, **kwargs):
        if (
            not self._closed
            and self._tip_ik_enabled
            and not bool(self.state.playing)
        ):
            self.scene.clear_tip_ik_override(render=False)
            self.ctrl.view_update()
        await super()._toggle_play(**kwargs)

    def _layer_callback(self, state_key, layer_key):
        def on_change(**values):
            if self._closed:
                return
            self.scene.set_layer_visible(
                layer_key, bool(values.get(state_key)), render=False
            )
            with self.state:
                self.state.frame_label = self.scene.caption
            self.ctrl.view_update()
        return on_change

    async def close(self):
        if self._closed:
            return
        self.scene.set_transform_start_callback(None)
        self.scene.set_tip_ik_status_callback(None)
        if getattr(self, "_tip_ik_enabled", False):
            self.scene.enable_tip_ik(False, render=False)
        reset_button = getattr(self, "_object_reset_button", None)
        reset_handler = getattr(self, "_object_reset_handler", None)
        if reset_button is not None:
            if reset_handler is not None:
                reset_button.on_click(reset_handler, remove=True)
            reset_button.close()
        interaction_help = getattr(self, "_interaction_help", None)
        if interaction_help is not None:
            interaction_help.close()
        tip_ik_status = getattr(self, "_tip_ik_status", None)
        if tip_ik_status is not None:
            tip_ik_status.close()
        handler = getattr(self, "_external_view_handler", None)
        for control in getattr(self, "_external_controls", ()):
            if handler is not None:
                control.unobserve(handler, names="value")
            persist = getattr(control, "_rtx_persist_handler", None)
            if persist is not None:
                control.unobserve(persist, names="value")
            control.close()
        for row in getattr(self, "_external_control_rows", ()):
            row.close()
        panel = getattr(self, "_external_control_panel", None)
        if panel is not None:
            panel.close()
        notebook_panel = getattr(self, "_notebook_panel", None)
        if notebook_panel is not None:
            notebook_panel.children = ()
            notebook_panel.close()
        notebook_iframe = getattr(self, "_notebook_iframe", None)
        if notebook_iframe is not None:
            notebook_iframe.close()
        self._external_controls = ()
        self._external_control_rows = ()
        self._external_control_panel = None
        self._external_view_handler = None
        self._object_reset_button = None
        self._object_reset_handler = None
        self._interaction_help = None
        self._tip_ik_status = None
        self._notebook_panel = None
        self._notebook_iframe = None
        await super().close()


async def close_live(global_name):
    old = globals().get(global_name)
    old_scene = getattr(old, "scene", None)
    if old is not None:
        await old.close()
        globals()[global_name] = None
    scene_global = global_name.removesuffix("_rtx") + "_scene"
    if old_scene is not None and globals().get(scene_global) is old_scene:
        globals()[scene_global] = None


async def launch_live(
    global_name, scene, layers, *, title, scene_name,
    height=680, frame_ms=125, lim=300.0, step=10.0,
    display_now=True, enable_tip_ik=False,
):
    app = None
    try:
        app = WeldingRTXLiveWidget(
            scene,
            layers=layers,
            title=title,
            layer_title="Graphic elements",
            layer_subtitle=(
                "RTX actor groups; instantaneous switches peak ↔ T"
            ),
            help_text=(
                "Camera: drag to orbit · middle-drag to pan · "
                "right-drag/wheel to zoom. Objects: click to select · "
                "Shift+left-drag to translate · "
                "Ctrl+left-drag to rotate."
                + (
                    " TCP: left-drag the orange handle for Mink IK."
                    if enable_tip_ik else ""
                )
            ),
            frame_ms=frame_ms,
        )
        app._tip_ik_enabled = bool(enable_tip_ik)
        app._tip_ik_status = widgets.HTML(
            layout=widgets.Layout(width="100%", min_height="22px")
        )
        if app._tip_ik_enabled:
            scene.set_tip_ik_status_callback(app._set_tip_ik_status)
            scene.enable_tip_ik(True, render=False)
        scene.set_transform_start_callback(
            app._pause_for_object_drag
        )
        globals()[global_name] = app
        controls = bind_live_controls(
            app, scene, scene_name, layers, lim=lim, step=step
        )
        if display_now:
            await app.display_cell(height=height)
            display(controls)
        else:
            height_css = (
                f"{height}px"
                if isinstance(height, (int, float)) else str(height)
            )
            app.ui.iframe_style = (
                "border: none; width: 100%; "
                f"height: {height_css};"
            )
            await app.ui.ready
            iframe = app.ui.ipywidget
            iframe.layout = widgets.Layout(
                width="100%", height=height_css
            )
            app._notebook_iframe = iframe
            app._notebook_panel = widgets.VBox(
                (iframe, controls),
                layout=widgets.Layout(width="100%"),
            )
        return app
    except Exception:
        globals()[global_name] = None
        if app is not None:
            await app.close()
        elif not scene._closed:
            scene.close()
        scene_global = global_name.removesuffix("_rtx") + "_scene"
        if globals().get(scene_global) is scene:
            globals()[scene_global] = None
        raise


arm_layers = (
    layer_spec("arm", "robot", "UR5e robot"),
    layer_spec("arm", "workpiece", "Workpiece"),
    layer_spec("arm", "seam", "Seam centreline"),
    layer_spec("arm", "executed_path", "Executed TCP path"),
    layer_spec("arm", "thermal", "End-of-weld fusion field"),
    layer_spec(
        "arm", "instant_mode",
        "Instantaneous field (off: peak history)", False,
    ),
    layer_spec("arm", "axes", "World axes"),
)
arm_scene = RobotRTXScene(
    [2.5],
    layers=arm_layers,
    current_frame=0,
    grow_trace=True,
    pool_model=finished_pool(),
)
arm_rtx = await launch_live(
    "arm_rtx",
    arm_scene,
    arm_layers,
    title="UR5e robot weave — NVIDIA RTX",
    scene_name="arm",
    height=690,
)


HTML(value='<iframe id="rtx-live-630bbdf6e6_trame__template_main" src="http://localhost:46515/index.html?ui=ma…

## 3. 时间滑块：沿摆动轨迹播放机械臂

`RTXLiveWidget` 自带的整数帧滑块映射到 0–5 s 的 41 个时刻
（步长 0.125 s），工具栏播放按钮按 8 fps 前进。红色执行轨迹随
时间增长，可观察 wrist 和枪尖随三角摆左右运动。与原 notebook 的
`ipywidgets.interactive_output` 不同，换帧只修改持久 actor，
不重建 Plotter/iframe，因此鼠标相机保持不变。

视图、机械臂透明度、当前时刻和图层状态仍持久化。§4 完成时会把热场
原位接入这个已显示的 scene，无需重跑本格；随后可切换焊末 `g.T`
瞬时池与 `g.peak` 熔合区历史。把滑块拖到
5 s 时枪尖与瞬时池头部对齐。


In [4]:
%%rtx_only 3
await close_live("pose_rtx")
pose_times = np.linspace(0.0, T_TRACK, 41)
pose_initial = int(np.argmin(
    np.abs(pose_times - float(state_value("pose", "t", 2.5)))
))
pose_layers = (
    layer_spec("pose", "robot", "UR5e robot"),
    layer_spec("pose", "workpiece", "Workpiece"),
    layer_spec("pose", "seam", "Seam centreline"),
    layer_spec("pose", "executed_path", "Executed path to current time"),
    layer_spec("pose", "thermal", "End-of-weld fusion field"),
    layer_spec(
        "pose", "instant_mode",
        "Instantaneous field (off: peak history)", False,
    ),
    layer_spec("pose", "axes", "World axes"),
)
pose_scene = RobotRTXScene(
    pose_times,
    layers=pose_layers,
    current_frame=pose_initial,
    grow_trace=True,
    pool_model=finished_pool(),
    size=(900, 600),
)
pose_rtx = await launch_live(
    "pose_rtx",
    pose_scene,
    pose_layers,
    title="UR5e executed weave playback — NVIDIA RTX",
    scene_name="pose",
    height=670,
    frame_ms=125,
)


HTML(value='<iframe id="rtx-live-f929321ca5_trame__template_main" src="http://localhost:33133/index.html?ui=ma…

## 4. 机器人执行轨迹 → 熔池

把 §1 的实际 TCP 轨迹经 `RobotExecutedWeave` 注入 `GoldakFDM`
(数据库中位工况, **超细网格** `solver=xfine`; `amplitude_m > 0`
自动切换全宽网格), 求解 5 s 瞬态温度场。本格约 30 s。

为什么要 0.5 mm 网格: 峰值场熔合面的"鱼鳞"棱脊节距是物理的
(每半摆动周期一道, v/2f ≈ 1.3 mm), 但 fine (0.8 mm) 下一个摆动
周期只有 ~3 格 — 接近网格 Nyquist, marching-cubes 把棱脊渲染成
夸大的分离"硬币"棱片, 且间距被网格拍频调制 (1.6–4 mm 不等)。
xfine 下节距 ~5 格, 棱脊变成平滑扇贝纹, 间距回到物理值。

In [5]:
%%rtx_only 4
rw = RobotExecutedWeave.from_tracking(t_tr, tip, P0, V_WELD,
                                      frequency_Hz=weave.frequency_Hz)
g = instantiate(cfg.goldak, Q=Q_ARC, weave=rw)
g.run(t_end=cfg.run.goldak.t_end, x_start=cfg.run.goldak.x_start)
L, W, D = g.pool_size()
print(f"{rw.describe()}")
print(f"网格 {g.Nx}x{g.Ny}x{g.Nz} (全宽), 熔池 L×W×D = {L:.1f}×{W:.1f}×{D:.1f} mm, "
      f"T_max = {g.peak.max():.0f} K")

# Run All 时 §2/§3 已经显示；把新求得的热场原位接入，不重建相机。
for _scene_name, _app_name in (
    ("arm_scene", "arm_rtx"), ("pose_scene", "pose_rtx")
):
    _scene = globals().get(_scene_name)
    _app = globals().get(_app_name)
    if _scene is not None and not _scene._closed:
        _scene.attach_pool(g, render=False)
        if _app is not None and not _app._closed:
            with _app.state:
                _app.state.frame_label = _scene.caption
            _app.ctrl.view_update()

机器人执行 (501 采样) 2 Hz × 实际 4.09 mm
网格 200x99x40 (全宽), 熔池 L×W×D = 18.5×11.0×5.0 mm, T_max = 5835 K


## 4b. 熔池对流修正 (模块 10A 耦合)

同一执行轨迹再解一遍**对流增强**温度场 (`convection=effective`,
方案 A, 见 `docs/melt_convection_assessment.md`): 池内导热率放大
`α_eff/α` 倍 (10A 由 `dγ/dT` 折算, 限幅 6), 把 Marangoni 搅拌近似为
增强热输运。dt 随最大扩散率缩小 (×6 步数), 本格约 20–30 s。
预期: 峰值温度从非物理的 ~5700 K (超沸点) 回落到沸点以下, 池内混合
把"鱼鳞"棱脊抹平约 5 倍 (顶面熔线 mean|ΔT| 182 → 38 K)。

对流场特意留在 `solver=fine` (0.8 mm): 池内混合本来就把棱脊抹平了,
xfine 对它没有视觉收益, 却要 ~5 分钟 (×12 网格代价再 ×6 步数)。
本格同时保存 0–5 s 的 float32 温度快照供 §5/§5b 共用；传导动画也用
`fine` 重解一次，§4 的 `xfine` 终态分析和定量结果保持不变。

In [ ]:
%%rtx_only 5
# 重跑本格前先释放引用旧动画缓存的下游场景。
close_weld_output_tabs()
await close_live("composite_rtx")
await close_live("seam_rtx")
snaps = conv_snaps = None

# 40 个 125 ms 区间 + t=0 环境温度首帧；§5 和 §5b 共用同一组快照。
N_FRAMES = 41
weld_frame_times = np.linspace(0.0, T_TRACK, N_FRAMES)

# 动画用 fine 传导场；§4 的 g 仍保留 xfine 终态精度。
cfg_gif = compose_cfg("sim_3d", "process=db_median", "solver=fine")
x_start = float(cfg_gif.run.goldak.x_start)
g_gif = instantiate(cfg_gif.goldak, Q=Q_ARC, weave=rw)
snaps = g_gif.run_with_snapshots(
    t_end=T_TRACK,
    x_start=x_start,
    frame_times=weld_frame_times,
    snapshot_dtype=np.float32,
)

# convection=effective: goldak 节点嵌套实例化 10A 对象 (dgamma_dT 插值自母材)
cfg_conv = compose_cfg("sim_3d", "process=db_median", "solver=fine",
                       "convection=effective")
g_conv = instantiate(cfg_conv.goldak, Q=Q_ARC, weave=rw)
conv_x_start = float(cfg_conv.run.goldak.x_start)
if not np.isclose(x_start, conv_x_start):
    raise ValueError("传导/对流动画必须使用相同的 x_start")
conv_snaps = g_conv.run_with_snapshots(
    t_end=T_TRACK,
    x_start=conv_x_start,
    frame_times=weld_frame_times,
    snapshot_dtype=np.float32,
)
if (
    len(snaps) != len(conv_snaps)
    or not np.array_equal(
        [sample[0] for sample in snaps],
        [sample[0] for sample in conv_snaps],
    )
):
    raise RuntimeError("传导/对流动画帧未对齐")

Lc, Wc, Dc = g_conv.pool_size()
print(f"对流增强 alpha_eff/alpha = {g_conv.k_pool_mult:.1f} "
      f"(dgamma/dT = {g_conv.convection.dgamma_dT:+.1e} N/(m K), 外向流)")
print(f"熔池 L×W×D = {Lc:.1f}×{Wc:.1f}×{Dc:.1f} mm (传导 {L:.1f}×{W:.1f}×{D:.1f}), "
      f"T_max = {g_conv.peak.max():.0f} K (传导 {g.peak.max():.0f} K)")
cache_mib = sum(
    field.nbytes
    for frames in (snaps, conv_snaps)
    for _, *fields in frames
    for field in fields
) / 1024**2
print(f"RTX 动画缓存: {len(snaps)} 帧 × 传导/对流 = {cache_mib:.0f} MiB")

## 5. 合成场景：机械臂 + 动态焊接熔池

温度场从 FDM 域映射进机器人基座系（焊缝起点 `P0` ↔ 热源起点
`x_start`，深度朝机器人坐标的 −z），并与 UR5e 姿态和实际 TCP 轨迹
逐帧同步。按 **Play** 可回放 0–5 s 焊接过程，也可拖动时间滑块：

- 红色等温面：传导场 `T ≥ Tm`；
- 黄色半透明面：HAZ `T ≥ 1073 K`；
- 顶面：`jet` 温度切片和与当前模式同步的标尺；
- 蓝色面：可选 10A 对流增强熔池。

图层菜单中的 **instantaneous field** 是一个互斥模式开关：
关闭显示截至当前帧的 `peak`（曾经熔过的长拖尾/鱼鳞包络），打开显示
当前帧 `T`（枪尖处的当下熔池）；melt/HAZ/slice/scalar bar
整组同步切换。**10A convection overlay** 独立开关，打开时传导
熔池自动变半透明，以便看到更短、更光滑的蓝色对流修正池。

所有 actor 和 VTK filter 都留在同一个经过 RTX 验证的 render window
中；播放只更新持久温度数组、机械臂姿态和增长轨迹，切模式、对流叠加
或机械臂透明度都不会丢失鼠标视角。

本格先启动并保留复合 RTX 场景，不单独显示输出；运行下一格后，它会与
焊缝 RTX 回放一起进入 tab，生成的 GIF 位于第一个默认页。


In [ ]:
%%rtx_only 6
close_weld_output_tabs()
await close_live("composite_rtx")
await close_live("seam_rtx")

class CompositeRTXScene(WeldingRTXSceneBase):
    """§5 动态复合场：热场、UR5e 与执行轨迹逐帧同步。"""

    def __init__(
        self, conductive_frames, convection_frames, *,
        layers, current_frame=0,
    ):
        self.conductive_frames = list(conductive_frames)
        self.convection_frames = list(convection_frames)
        if len(self.conductive_frames) < 2:
            raise ValueError("composite playback requires at least two frames")
        self.frame_times = np.array(
            [sample[0] for sample in self.conductive_frames],
            dtype=float,
        )
        convection_times = np.array(
            [sample[0] for sample in self.convection_frames],
            dtype=float,
        )
        if not np.array_equal(self.frame_times, convection_times):
            raise ValueError("conductive/convection frame times differ")
        if any(
            not np.array_equal(getattr(g_gif, axis), getattr(g_conv, axis))
            for axis in ("x", "y", "z")
        ):
            raise ValueError("animation grids must share coordinates")
        self._conductive_melt = []
        gx = (g_gif.x - x_start + P0[0]) * 1e3
        gz = (P0[2] - g_gif.z) * 1e3
        focal = np.array([gx.mean(), 0.0, gz.max()])
        super().__init__(
            layers=layers,
            frame_count=len(self.frame_times),
            current_frame=current_frame,
            camera_home=(
                tuple(focal + [210, -320, 230]),
                tuple(focal),
                (0.0, 0.0, 1.0),
            ),
            size=(1000, 640),
            floor=(
                focal[0] - 260.0, focal[0] + 260.0,
                -210.0, 210.0, gz.min() - 12.0, 25.0,
            ),
        )

    def _build_scene(self):
        self.robot = RobotRig(self, arm)
        self.trace = PolylineRig(
            self, "executed_path", "#d81b1b", radius=0.6
        )
        self.add_axes(
            origin=(P0[0] * 1e3 - 80, -60, P0[2] * 1e3 - 20),
            length=35.0,
            transform_group="process",
        )

        _, cond_inst0, cond_peak0 = self.conductive_frames[0]
        _, conv_inst0, conv_peak0 = self.convection_frames[0]
        (
            self.grid,
            gx,
            _,
            gz,
            self.buffers,
            self.vtk_arrays,
        ) = goldak_grid(
            g_gif,
            {
                "cond_peak": cond_peak0,
                "cond_inst": cond_inst0,
                "conv_peak": conv_peak0,
                "conv_inst": conv_inst0,
            },
            dynamic=True,
            x_start=x_start,
        )
        outline = vtk.vtkOutlineFilter()
        outline.SetInputData(self.grid)
        self.add_surface(
            outline, "domain", color="#777777", opacity=0.8,
            smooth=False,
        )
        clim = (
            float(g_gif.T0),
            float(np.max(g_gif.peak)),
        )
        top = slice_filter(
            self.grid, (gx.mean(), 0.0, gz.max() - 1e-3)
        )
        convection_visible = self._layer_visible.get(
            "convection", False
        )
        for field, mode, label in (
            ("cond_peak", "peak", "peak"),
            ("cond_inst", "instant", "instant"),
        ):
            melt_record, _ = self.add_surface(
                contour_filter(self.grid, field, float(g_gif.Tm)),
                "thermal",
                mode=mode,
                color="#d7191c",
                opacity=0.35 if convection_visible else 0.90,
            )
            self._conductive_melt.append(melt_record)
            self.add_surface(
                contour_filter(self.grid, field, 1073.0),
                "thermal",
                mode=mode,
                color="#ffd92f",
                opacity=0.25,
            )
            _, lut = self.add_surface(
                top,
                "thermal",
                mode=mode,
                scalars=field,
                cmap="jet",
                clim=clim,
                opacity=0.80,
            )
            self.add_scalar_bar(
                lut, f"T {label} [K]", "thermal", mode=mode
            )

        for field, mode in (
            ("conv_peak", "peak"),
            ("conv_inst", "instant"),
        ):
            self.add_surface(
                contour_filter(
                    self.grid, field, float(g_conv.Tm)
                ),
                "convection",
                mode=mode,
                color="#2459b3",
                opacity=0.95,
            )

    def _update_frame(self, index):
        t_now, cond_inst, cond_peak = self.conductive_frames[index]
        _, conv_inst, conv_peak = self.convection_frames[index]
        for field, values in (
            ("cond_inst", cond_inst),
            ("cond_peak", cond_peak),
            ("conv_inst", conv_inst),
            ("conv_peak", conv_peak),
        ):
            np.copyto(
                self.buffers[field],
                np.asarray(values).ravel(order="F"),
                casting="unsafe",
            )
            self.vtk_arrays[field].Modified()
        self.grid.GetPointData().Modified()
        self.grid.Modified()
        self._update_robot_pose(q_at(t_now))
        mask = t_tr <= t_now
        self.trace.update(tip[mask] * 1e3 + [0, 0, 0.3])
        self._update_caption(t_now)

    def _update_caption(self, t_now):
        mode = "instantaneous T" if self._instant_mode else "peak history"
        suffix = (
            " + 10A convection"
            if self._layer_visible.get("convection", False)
            else ""
        )
        self.caption = (
            f"t = {float(t_now):.2f} s · robot-executed pool · "
            f"{mode}{suffix}"
        )

    def _on_layer_changed(self, key, visible):
        if key == "convection":
            for record in self._conductive_melt:
                record["opacity"] = 0.35 if visible else 0.90
        if key in {"convection", "instant_mode"}:
            self._update_caption(self.frame_times[self.current_frame])


composite_layers = (
    layer_spec("scene5", "robot", "UR5e robot"),
    layer_spec("scene5", "ik_handle", "TCP IK handle"),
    layer_spec("scene5", "domain", "Goldak grid outline"),
    layer_spec(
        "scene5", "executed_path", "Executed TCP path to current time"
    ),
    layer_spec("scene5", "thermal", "Conductive melt / HAZ / top slice"),
    layer_spec(
        "scene5", "instant_mode",
        "Instantaneous field (off: peak history)", False,
    ),
    layer_spec(
        "scene5", "convection",
        "10A convection overlay", False,
    ),
    layer_spec("scene5", "axes", "Local axes"),
)
composite_initial = int(np.argmin(np.abs(
    weld_frame_times - float(state_value("scene5", "t", 0.0))
)))
composite_scene = CompositeRTXScene(
    snaps,
    conv_snaps,
    layers=composite_layers,
    current_frame=composite_initial,
)
composite_rtx = await launch_live(
    "composite_rtx",
    composite_scene,
    composite_layers,
    title="UR5e + robot-executed weld pool — NVIDIA RTX",
    scene_name="scene5",
    height=710,
    frame_ms=125,
    lim=500.0,
    step=20.0,
    display_now=False,
    enable_tip_ik=True,
)


## 5b. 动画：焊缝成形 GIF + RTX 实时回放

复用 §4b 为合成场景求得的 41 帧 fine-grid 传导快照（含 t=0 环境
温度首帧），不再重复求解 `GoldakFDM`：

- 枪尖处亮红：瞬时熔池 `T ≥ Tm`；
- 身后暗红：逐帧增长的已熔/凝固焊缝 `peak ≥ Tm`；
- 顶面：`T ≥ 400 K` 的 `inferno` 热晕；
- 机械臂与红色 TCP 轨迹同步增长。

动画使用 `solver=fine`（0.8 mm）控制内存和逐帧 contour 耗时。
渲染不创建 PyVista off-screen window：`SeamRTXScene` 持有一个已验证的
EGL context，`np.copyto` 更新两个持久数组后调用 `Modified()`；
服务端 VTK 重新计算 contour/cutter/threshold，随后由 RTX 光栅化。
相同 render window
既写出 `results/robot6_weave_seam.gif`，也交给 live widget 播放。

本格最终只显示一个三页 tab：**GIF animation**（默认）、
**Composite RTX** 和 **Seam RTX**。切换到任一 live 页时会主动刷新
刚刚解除隐藏的远程视图；离开 live 页会暂停其播放。
两个 live 页都可左拖橙色 TCP 手柄执行保持工具姿态的 Mink IK；
**TCP IK handle** 图层可独立隐藏。该手柄在 GIF 抓帧阶段保持禁用，
仅在 live widget 启动后显示。


In [ ]:
%%rtx_only 7
close_weld_output_tabs()
await close_live("seam_rtx")
if (
    globals().get("composite_rtx") is None
    or composite_rtx._closed
    or getattr(composite_rtx, "_notebook_panel", None) is None
):
    raise RuntimeError(
        "复合 RTX 场景尚未准备；请先运行单元格 7。"
    )

from PIL import Image as PILImage

GIF_PATH = REPO_ROOT / "results" / "robot6_weave_seam.gif"
GIF_PATH.parent.mkdir(exist_ok=True)

if not snaps:
    raise RuntimeError("快照为空；检查 Goldak 时间步和帧索引")


class SeamRTXScene(WeldingRTXSceneBase):
    """动态焊缝：持久 structured grid + topology-changing filters。"""

    def __init__(self, snapshots, *, layers):
        self.snapshots = list(snapshots)
        self.frame_times = np.array(
            [sample[0] for sample in snapshots], dtype=float
        )
        ggx = (g_gif.x - x_start + P0[0]) * 1e3
        ggz = (P0[2] - g_gif.z) * 1e3
        focal = np.array([
            P0[0] * 1e3 + 0.5 * V_WELD * T_TRACK * 1e3,
            0.0,
            ggz.max(),
        ])
        self._grid_x = ggx
        self._grid_z = ggz
        super().__init__(
            layers=layers,
            frame_count=len(self.snapshots),
            current_frame=0,
            camera_home=(
                tuple(focal + [95, -160, 120]),
                tuple(focal),
                (0.0, 0.0, 1.0),
            ),
            size=(880, 540),
            floor=(
                focal[0] - 260.0, focal[0] + 260.0,
                -210.0, 210.0, ggz.min() - 12.0, 25.0,
            ),
        )

    def _build_scene(self):
        self.robot = RobotRig(self, arm)
        add_workpiece(self, opacity=0.40)
        self.seam = PolylineRig(
            self, "seam", "#151515", radius=0.75
        )
        self.seam.update(seam_points())
        self.trace = PolylineRig(
            self, "executed_path", "#d81b1b", radius=0.8
        )
        self.time_text = self.add_text(
            "t = 0.00 s", "annotation",
            position=(16, 505), font_size=18, color="#f4f7fa",
        )
        self.add_text(
            "bright: molten pool | dark: solidified seam",
            "annotation", position=(495, 510),
            font_size=13, color="#e3e9ef",
        )
        t0, inst0, peak0 = self.snapshots[0]
        (
            self.grid,
            _,
            _,
            _,
            self.buffers,
            self.vtk_arrays,
        ) = goldak_grid(
            g_gif,
            {"inst": inst0, "peak": peak0},
            dynamic=True,
            x_start=x_start,
        )
        self.add_surface(
            contour_filter(self.grid, "peak", float(g_gif.Tm)),
            "history",
            color="#9e241a",
            opacity=0.90,
        )
        self.add_surface(
            contour_filter(self.grid, "inst", float(g_gif.Tm)),
            "pool",
            color="#ff5916",
            opacity=1.0,
        )
        top = slice_filter(
            self.grid,
            (
                self._grid_x.mean(),
                0.0,
                self._grid_z.max() - 1e-3,
            ),
        )
        hot_top = threshold_upper(top, "inst", 400.0)
        self.add_surface(
            hot_top,
            "halo",
            scalars="inst",
            cmap="inferno",
            clim=(float(g_gif.T0), 2600.0),
            opacity=0.85,
        )
        self.add_axes(
            origin=(
                P0[0] * 1e3 - 75, -55, P0[2] * 1e3 - 18
            ),
            length=28.0,
            transform_group="process",
        )

    def _update_frame(self, index):
        t_now, inst, peak = self.snapshots[index]
        np.copyto(
            self.buffers["inst"],
            np.asarray(inst).ravel(order="F"),
            casting="unsafe",
        )
        np.copyto(
            self.buffers["peak"],
            np.asarray(peak).ravel(order="F"),
            casting="unsafe",
        )
        for vtk_array in self.vtk_arrays.values():
            vtk_array.Modified()
        self.grid.GetPointData().Modified()
        self.grid.Modified()
        self._update_robot_pose(q_at(t_now))
        mask = t_tr <= t_now
        self.trace.update(tip[mask] * 1e3 + [0, 0, 0.3])
        self.time_text["prop"].SetInput(f"t = {t_now:.2f} s")
        self.caption = (
            f"t = {t_now:.2f} s · bright molten / dark history"
        )


seam_layers = (
    layer_spec("scene5b", "robot", "UR5e robot"),
    layer_spec("scene5b", "ik_handle", "TCP IK handle"),
    layer_spec("scene5b", "workpiece", "Workpiece"),
    layer_spec("scene5b", "seam", "Seam centreline"),
    layer_spec(
        "scene5b", "executed_path", "Executed path to current time"
    ),
    layer_spec("scene5b", "history", "Solidified / peak envelope"),
    layer_spec("scene5b", "pool", "Instantaneous molten pool"),
    layer_spec("scene5b", "halo", "Top-surface thermal halo"),
    layer_spec("scene5b", "annotation", "Time / colour legend"),
    layer_spec("scene5b", "axes", "Local axes"),
)
seam_scene = SeamRTXScene(snaps, layers=seam_layers)

# ---- 同一个 NVIDIA EGL render window -> Pillow GIF ----
gif_frames = []
for index in range(seam_scene.frame_count):
    seam_scene.set_frame(index, render=False)
    gif_frames.append(PILImage.fromarray(seam_scene.capture_rgb()))
seam_scene.set_frame(0, render=False)
gif_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=gif_frames[1:],
    loop=0,
    duration=[125] * (len(gif_frames) - 1) + [1500],
    optimize=True,
)
gif_summary = (
    f"{GIF_PATH.relative_to(REPO_ROOT)} · {len(gif_frames)} 帧 · "
    f"{GIF_PATH.stat().st_size / 1024:.0f} kB · "
    f"{seam_scene.backend_info.label}"
)

seam_rtx = await launch_live(
    "seam_rtx",
    seam_scene,
    seam_layers,
    title="Robot weave seam formation — NVIDIA RTX",
    scene_name="scene5b",
    height=650,
    frame_ms=125,
    lim=180.0,
    step=10.0,
    display_now=False,
    enable_tip_ik=True,
)

weld_gif_image = widgets.Image(
    value=GIF_PATH.read_bytes(),
    format="gif",
    layout=widgets.Layout(
        width="880px", max_width="100%", height="auto"
    ),
)
weld_gif_caption = widgets.HTML(
    value=(
        "<div style='text-align:center;color:#4b5563;padding-top:6px'>"
        f"{gif_summary}</div>"
    )
)
weld_gif_panel = widgets.VBox(
    (weld_gif_image, weld_gif_caption),
    layout=widgets.Layout(
        width="100%", align_items="center", padding="12px 0"
    ),
)
weld_output_tabs = widgets.Tab(
    children=(
        weld_gif_panel,
        composite_rtx._notebook_panel,
        seam_rtx._notebook_panel,
    ),
    selected_index=0,
    layout=widgets.Layout(width="100%"),
)
for index, title in enumerate((
    "GIF animation", "Composite RTX", "Seam RTX"
)):
    weld_output_tabs.set_title(index, title)


def _on_weld_output_tab(
    change, apps=(composite_rtx, seam_rtx),
):
    selected = change["new"]
    for index, app in enumerate(apps, start=1):
        if index != selected and bool(app.state.playing):
            with app.state:
                app.state.playing = False
    app = apps[selected - 1] if selected in (1, 2) else None
    if app is not None and not app._closed:
        # iframe 初始位于隐藏 tab；首次展示后强制发送正确尺寸的帧。
        app.ctrl.view_update()


weld_output_tabs.observe(
    _on_weld_output_tab, names="selected_index"
)
weld_output_tabs._rtx_selection_handler = _on_weld_output_tab
display(weld_output_tabs)


2026-07-31 19:05:28.068 (2379.791s) [    7B365072F740]vtkMatplotlibMathTextUt:954   WARN| vtkMatplotlibMathTextUtilities (0x103b0a70): Text shadow is not supported with math text.


## 相关

- **RTX 前端**：本 notebook 的 `WeldingRTXSceneBase` /
  `RobotRTXScene` / `CompositeRTXScene` / `SeamRTXScene` 是
  `vendor/trame-rtx-widget` 的应用侧 `FrameScene` 实现。
  `PlotlyFrameScene` 只适合固定拓扑动画，不用于 Goldak 等值面。
- **原 PyVista 自包含版本与工具箱**：
  `robot6_weave_interactive_demo.ipynb` 和 `notebooks/utils/`
  保留 `html_view`、客户端图层切换、`WidgetStore`、
  `solve_with_snapshots` 及独立 GIF 脚本。本 RTX notebook 保留
  物理计算和视图持久化语义，但渲染生命周期由 live widget 管理。
- **定量部署分析**（摆频扫描 0.5–4 Hz 幅值保真度、梯形波超调、
  理想 vs 执行熔池）：`robot_deployment_scenarios.ipynb`。
- **熔池对流**：耦合路线见 `docs/melt_convection_assessment.md`；
  三个 Marangoni 降阶模型（10A/10B/10C）的独立演示见
  `marangoni_comparison.ipynb`。
- **五个典型工况**：`welding_scenarios_interactive_demo.ipynb`；
  PyVista 内联后端技巧：`pyvista_interactive_demo.ipynb`。
- **CLI**：机械臂演示 `uv run welding-sim-vi`（模块 7b 输出
  `m7b_robot6_vi.png`）；热场
  `uv run welding-sim-3d process=db_median weave=triangle solver=fine`
  （加 `convection=effective` 得 §4b）。
- **玩法**：把 §1 的 `compose_cfg("sim_3d", "weave=triangle")`
  改成 `weave=pattern1` 看梯形摆超调；改 `P0` / `R_REF` 放置
  立焊/横焊。注意当前热场是传导 + 有效导热率近似，不含重力或焊接
  位置依赖，因此位置只改变机器人侧几何，不改变熔池物理。可换回默认
  球腕臂或修改 `conf/model/robot6_ur5e.yaml`；`SixDofArm` 接受任意
  标准 DH 表（`dh_d/dh_a/dh_alpha_deg`）。

### Composite RTX 与 Seam RTX 的区别

二者是同一组 41 帧焊接过程的不同视图，并非两次独立仿真；都同步
显示 UR5e 运动和实际执行的 TCP 轨迹。

| Composite RTX | Seam RTX |
|---|---|
| 面向热模型检查的工程视图 | 面向焊缝成形观察的过程视图 |
| 显示 Goldak 计算域轮廓 | 显示实体工件和焊缝中心线 |
| 显示传导熔池、HAZ、温度切片及标尺 | 同时显示亮色瞬时熔池、暗色凝固历史和顶面热晕 |
| 可在瞬时温度与峰值历史间切换 | 瞬时熔池与累计焊缝始终同时显示 |
| 可叠加 10A 对流修正场 | 只使用传导快照 |
| 宽视角覆盖机械臂和热计算域 | 近视角聚焦焊缝成形，也是 GIF 的渲染场景 |

简言之：**Composite RTX 用于检查热模型，Seam RTX 用于观察焊缝如何形成。**

完成后可显式释放全部 live resources：

```python
close_weld_output_tabs()
for app in (arm_rtx, pose_rtx, composite_rtx, seam_rtx):
    if app is not None:
        await app.close()
```
